<a href="https://colab.research.google.com/github/Al-Amin95/PromptInjectionDetectionSystem/blob/data-analysis/notebooks/1-data-analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1 Data Analysis: Prompt Injection Detection Dataset

## Part-1:  Notebook Setup and Reproducibility

In [ ]:
# 1. Mount Google Drive

from google.colab import drive
from pathlib import Path

DRIVE_ROOT = Path("/content/drive")
MY_DRIVE = DRIVE_ROOT / "MyDrive"

if MY_DRIVE.exists():
    print("Google Drive is already mounted.")
else:
    drive.mount("/content/drive")
    print("Google Drive mounted successfully.")

print("MyDrive path:", MY_DRIVE)
print("MyDrive exists:", MY_DRIVE.exists())

In [ ]:
# 2-Auto-detect project folder path

from pathlib import Path

MY_DRIVE = Path("/content/drive/MyDrive")

print("MyDrive exists:", MY_DRIVE.exists())

# 1. Find USW dissertation folder automatically
usw_candidates = []

for item in MY_DRIVE.iterdir():
    if item.is_dir():
        normalized_name = item.name.lower().replace("_", " ").replace("-", " ")
        if "usw" in normalized_name and "dissertation" in normalized_name:
            usw_candidates.append(item)

print("\nUSW candidate folders:")
for idx, folder in enumerate(usw_candidates):
    print(idx, "->", repr(folder.name), "| full path:", folder)

if len(usw_candidates) == 0:
    raise FileNotFoundError("No USW Dissertation folder found inside MyDrive.")

USW_DIR = usw_candidates[0]

# 2. Find project folder automatically
project_candidates = []

for item in USW_DIR.iterdir():
    if item.is_dir():
        normalized_name = item.name.lower().replace("_", " ").replace("-", " ")
        if "prompt" in normalized_name and "injection" in normalized_name:
            project_candidates.append(item)

print("\nProject candidate folders:")
for idx, folder in enumerate(project_candidates):
    print(idx, "->", repr(folder.name), "| full path:", folder)

if len(project_candidates) == 0:
    raise FileNotFoundError("No Prompt Injection project folder found inside the USW folder.")

PROJECT_ROOT = project_candidates[0]

# 3. Find Notebooks folder automatically, case-insensitive
notebook_candidates = []

for item in PROJECT_ROOT.iterdir():
    if item.is_dir() and item.name.lower() == "notebooks":
        notebook_candidates.append(item)

print("\nNotebook candidate folders:")
for idx, folder in enumerate(notebook_candidates):
    print(idx, "->", repr(folder.name), "| full path:", folder)

if len(notebook_candidates) == 0:
    raise FileNotFoundError("No Notebooks folder found inside the project folder.")

NOTEBOOKS_DIR = notebook_candidates[0]

print("\nFINAL PATHS")
print("USW_DIR:", USW_DIR)
print("PROJECT_ROOT:", PROJECT_ROOT)
print("NOTEBOOKS_DIR:", NOTEBOOKS_DIR)

print("\nCHECKS")
print("USW_DIR exists:", USW_DIR.exists())
print("PROJECT_ROOT exists:", PROJECT_ROOT.exists())
print("NOTEBOOKS_DIR exists:", NOTEBOOKS_DIR.exists())

In [ ]:
# 3. GitHub connection check and clone/pull public data-analysis branch

from pathlib import Path
import subprocess
import shutil

USE_GITHUB = True

GITHUB_USERNAME = "Al-Amin95"
GITHUB_REPOSITORY = "PromptInjectionDetectionSystem"
GITHUB_BRANCH = "data-analysis"
GITHUB_NOTEBOOK_PATH = "notebooks/1-data-analysis.ipynb"

GITHUB_REPO_URL = f"https://github.com/{GITHUB_USERNAME}/{GITHUB_REPOSITORY}"
GITHUB_CLONE_URL = f"{GITHUB_REPO_URL}.git"

REPO_LOCAL_PATH = Path("/content/PromptInjectionDetectionSystem")

print("GitHub connection information")
print("Repository:", GITHUB_REPO_URL)
print("Branch:", GITHUB_BRANCH)
print("Notebook path:", GITHUB_NOTEBOOK_PATH)
print("Repository visibility: Public")

def run_command(command, cwd=None):
    result = subprocess.run(
        command,
        cwd=cwd,
        capture_output=True,
        text=True
    )

    if result.stdout.strip():
        print(result.stdout.strip())

    if result.returncode != 0:
        if result.stderr.strip():
            print(result.stderr.strip())
        raise RuntimeError(f"Command failed: {' '.join(command)}")

if REPO_LOCAL_PATH.exists() and not (REPO_LOCAL_PATH / ".git").exists():
    print("\nFolder exists but is not a Git repository. Removing it.")
    shutil.rmtree(REPO_LOCAL_PATH)

if REPO_LOCAL_PATH.exists():
    print("\nRepository exists. Pulling latest changes.")
    run_command(["git", "remote", "set-url", "origin", GITHUB_CLONE_URL], cwd=REPO_LOCAL_PATH)
    run_command(["git", "fetch", "origin"], cwd=REPO_LOCAL_PATH)
    run_command(["git", "checkout", GITHUB_BRANCH], cwd=REPO_LOCAL_PATH)
    run_command(["git", "pull", "origin", GITHUB_BRANCH], cwd=REPO_LOCAL_PATH)
else:
    print("\nRepository not found. Cloning repository.")
    run_command(
        ["git", "clone", "-b", GITHUB_BRANCH, GITHUB_CLONE_URL, str(REPO_LOCAL_PATH)],
        cwd="/content"
    )

current_branch = subprocess.run(
    ["git", "branch", "--show-current"],
    cwd=REPO_LOCAL_PATH,
    capture_output=True,
    text=True,
    check=True
).stdout.strip()

print("\nGitHub repository ready.")
print("Repository local path:", REPO_LOCAL_PATH)
print("Repository exists:", REPO_LOCAL_PATH.exists())
print("Current Git branch:", current_branch)

if current_branch != GITHUB_BRANCH:
    raise ValueError(f"Wrong branch. Expected {GITHUB_BRANCH}, but got {current_branch}.")

In [ ]:
# 4-Verify GitHub branch and raw dataset files

RAW_DATA_DIR = REPO_LOCAL_PATH / "data" / "raw"

current_branch = subprocess.run(
    ["git", "branch", "--show-current"],
    cwd=REPO_LOCAL_PATH,
    capture_output=True,
    text=True,
    check=True
).stdout.strip()

raw_files = sorted([file for file in RAW_DATA_DIR.iterdir() if file.is_file()])
parquet_files = [file for file in raw_files if file.suffix == ".parquet"]

print("Repository local path:", REPO_LOCAL_PATH)
print("Repository exists:", REPO_LOCAL_PATH.exists())
print("Current Git branch:", current_branch)
print("Raw data directory:", RAW_DATA_DIR)
print("RAW_DATA_DIR exists:", RAW_DATA_DIR.exists())
print("Number of raw files:", len(raw_files))
print("Number of parquet dataset files:", len(parquet_files))

print("\nRaw dataset files:")
for file in raw_files:
    print(file.name)

if current_branch != GITHUB_BRANCH:
    raise ValueError(f"Wrong branch. Expected {GITHUB_BRANCH}, but got {current_branch}.")

if len(parquet_files) < 2:
    raise FileNotFoundError("Expected train and test parquet files were not found in data/raw.")

print("\nGitHub branch and raw dataset verification completed successfully.")

In [ ]:
# 5-Import libraries for data analysis

import os
import sys
import re
import json
import random
import platform
from pathlib import Path
from datetime import datetime
from collections import Counter

import numpy as np
import pandas as pd

import matplotlib
import matplotlib.pyplot as plt

print("Libraries imported successfully.")

In [ ]:
# 6-Set random seed

SEED = 42

random.seed(SEED)
np.random.seed(SEED)

print(f"Random seed set to: {SEED}")

In [ ]:
# 7-Record environment information

environment_info = {
    "python_version": sys.version,
    "platform": platform.platform(),
    "numpy_version": np.__version__,
    "pandas_version": pd.__version__,
    "matplotlib_version": matplotlib.__version__,
    "notebook_run_datetime": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "random_seed": SEED
}

for key, value in environment_info.items():
    print(f"{key}: {value}")

In [ ]:
# 8-Define project paths


DATA_DIR = PROJECT_ROOT / "data"
RAW_DATA_DIR = DATA_DIR / "raw"
PROCESSED_DATA_DIR = DATA_DIR / "processed"

OUTPUTS_DIR = PROJECT_ROOT / "outputs"
DATA_ANALYSIS_OUTPUT_DIR = OUTPUTS_DIR / "data_analysis"

TABLES_DIR = DATA_ANALYSIS_OUTPUT_DIR / "tables"
FIGURES_DIR = DATA_ANALYSIS_OUTPUT_DIR / "figures"
REPORTS_DIR = DATA_ANALYSIS_OUTPUT_DIR / "reports"
LOGS_DIR = DATA_ANALYSIS_OUTPUT_DIR / "logs"

MODELS_DIR = PROJECT_ROOT / "models"
APP_DIR = PROJECT_ROOT / "webapp"

print("Project paths defined successfully.")
print("Project root:", PROJECT_ROOT)
print("Notebooks directory:", NOTEBOOKS_DIR)
print("Raw data directory:", RAW_DATA_DIR)
print("Processed data directory:", PROCESSED_DATA_DIR)
print("Tables directory:", TABLES_DIR)
print("Figures directory:", FIGURES_DIR)
print("Reports directory:", REPORTS_DIR)
print("Logs directory:", LOGS_DIR)
print("Models directory:", MODELS_DIR)
print("App directory:", APP_DIR)

In [ ]:
# 9-Create required folders

project_dirs = [
    PROJECT_ROOT,
    NOTEBOOKS_DIR,
    DATA_DIR,
    RAW_DATA_DIR,
    PROCESSED_DATA_DIR,
    OUTPUTS_DIR,
    DATA_ANALYSIS_OUTPUT_DIR,
    TABLES_DIR,
    FIGURES_DIR,
    REPORTS_DIR,
    LOGS_DIR,
    MODELS_DIR,
    APP_DIR
]

for directory in project_dirs:
    directory.mkdir(parents=True, exist_ok=True)

print("Required folders created/confirmed successfully.")

In [ ]:
# 10-Check folder structure

folder_check = []

for directory in project_dirs:
    folder_check.append({
        "folder_name": directory.name,
        "folder_path": str(directory),
        "exists": directory.exists()
    })

folder_check_df = pd.DataFrame(folder_check)

folder_check_df

In [ ]:
# 11-Save folder structure check

folder_check_path = LOGS_DIR / "folder_structure_check.csv"

folder_check_df.to_csv(folder_check_path, index=False)

print("Folder structure check saved to:")
print(folder_check_path)
print("File exists:", folder_check_path.exists())

In [ ]:
# 12-Save analysis_config.json

analysis_config = {
    "project_title": "Prompt Injection Detection Using Fine-Tuned Transformer Models: A Glass-Box Approach with Explainable AI",
    "notebook_name": "1-data-analysis.ipynb",
    "analysis_stage": "Part 1 - Data Analysis",
    "dataset_name": "deepset/prompt-injections",
    "task": "Binary prompt injection classification",
    "label_mapping": {
        "0": "SAFE",
        "1": "INJECTION"
    },
    "random_seed": SEED,
    "raw_data_policy": "Read-only in Part 1. No permanent modification in this notebook.",
    "github_repository": GITHUB_REPO_URL,
    "github_branch": GITHUB_BRANCH,
    "github_notebook_path": GITHUB_NOTEBOOK_PATH,
    "project_root": str(PROJECT_ROOT),
    "notebooks_dir": str(NOTEBOOKS_DIR),
    "raw_data_dir": str(RAW_DATA_DIR),
    "processed_data_dir": str(PROCESSED_DATA_DIR),
    "tables_dir": str(TABLES_DIR),
    "figures_dir": str(FIGURES_DIR),
    "reports_dir": str(REPORTS_DIR),
    "logs_dir": str(LOGS_DIR),
    "models_dir": str(MODELS_DIR),
    "app_dir": str(APP_DIR),
    "folder_structure_check_file": str(folder_check_path),
    "created_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "python_version": sys.version,
    "numpy_version": np.__version__,
    "pandas_version": pd.__version__,
    "matplotlib_version": matplotlib.__version__
}

config_path = LOGS_DIR / "analysis_config.json"

with open(config_path, "w", encoding="utf-8") as f:
    json.dump(analysis_config, f, indent=4)

print("Analysis configuration saved to:")
print(config_path)
print("File exists:", config_path.exists())

# Part-2: Raw Dataset Verification and Loading analysis

In [ ]:
# 1-Set pandas display options

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 300)
pd.set_option("display.width", 120)

print("Pandas display options configured successfully.")

In [ ]:
# 2-Set matplotlib defaults

plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["figure.dpi"] = 100
plt.rcParams["savefig.dpi"] = 300
plt.rcParams["axes.grid"] = True

print("Matplotlib defaults configured successfully.")

In [ ]:
# 3-Check raw data directory exists
print("Raw data directory:", RAW_DATA_DIR)
print("RAW_DATA_DIR exists:", RAW_DATA_DIR.exists())

In [ ]:
# 3-List files inside raw data directory

raw_files = sorted([file for file in RAW_DATA_DIR.iterdir() if file.is_file()])

print("Number of files in raw data directory:", len(raw_files))
print("\nRaw data files:")

for file in raw_files:
    print(file.name)

In [ ]:
# 4-Define raw train and test file paths

RAW_TRAIN_PATH = RAW_DATA_DIR / "train-00000-of-00001-9564e8b05b4757ab.parquet"
RAW_TEST_PATH = RAW_DATA_DIR / "test-00000-of-00001-701d16158af87368.parquet"

print("Raw train path:", RAW_TRAIN_PATH)
print("Raw test path:", RAW_TEST_PATH)

In [ ]:
# 5-Check raw train and test files exist

print("Raw train file exists:", RAW_TRAIN_PATH.exists())
print("Raw test file exists:", RAW_TEST_PATH.exists())

In [ ]:
# 6-Load raw train dataset

train_df = pd.read_parquet(RAW_TRAIN_PATH)

print("Raw train dataset loaded successfully.")
print("Train dataset shape:", train_df.shape)

In [ ]:
# 7-Load raw test dataset

test_df = pd.read_parquet(RAW_TEST_PATH)

print("Raw test dataset loaded successfully.")
print("Test dataset shape:", test_df.shape)

In [ ]:
# 8-Check train and test dataset shapes

print("Train dataset shape:", train_df.shape)
print("Test dataset shape:", test_df.shape)

print("\nTrain rows:", train_df.shape[0])
print("Train columns:", train_df.shape[1])

print("\nTest rows:", test_df.shape[0])
print("Test columns:", test_df.shape[1])

In [ ]:
# 9- Display first few rows of train and test datasets

print("First 5 rows of train dataset:")
display(train_df.head())

print("\nFirst 5 rows of test dataset:")
display(test_df.head())

In [ ]:
# 10-Check column names

print("Train dataset columns:")
print(train_df.columns.tolist())

print("\nTest dataset columns:")
print(test_df.columns.tolist())

In [ ]:
# 11-Check data types

print("Train dataset data types:")
print(train_df.dtypes)

print("\nTest dataset data types:")
print(test_df.dtypes)

#Part-3: Combined EDA Dataset Creation Analysis

In [ ]:
# 1-2 Create temporary train and test dataset copies with split labels

train_eda_df = train_df.copy()
test_eda_df = test_df.copy()

train_eda_df["split"] = "train"
test_eda_df["split"] = "test"

print("Temporary train EDA dataset created.")
print("Temporary test EDA dataset created.")
print("Train EDA shape:", train_eda_df.shape)
print("Test EDA shape:", test_eda_df.shape)

In [ ]:
# 3-Combine train and test only for exploratory data analysis

combined_eda_df = pd.concat([train_eda_df, test_eda_df], ignore_index=True)

print("Combined EDA dataset created successfully.")
print("Combined EDA shape:", combined_eda_df.shape)

In [ ]:
# 4-Check combined dataset shape

print("Combined EDA dataset shape:", combined_eda_df.shape)

print("\nSplit counts:")
print(combined_eda_df["split"].value_counts())

print("\nCombined EDA columns:")
print(combined_eda_df.columns.tolist())

In [ ]:
# 5-Confirm combined dataset is for EDA only

print("Phase 3 read-only confirmation:")
print("combined_eda_df is a temporary dataset for exploratory data analysis only.")
print("combined_eda_df will not be used directly for fine-tuning.")
print("Raw train and test datasets have not been modified.")
print("Original train_df shape:", train_df.shape)
print("Original test_df shape:", test_df.shape)
print("Combined EDA shape:", combined_eda_df.shape)

#Part-4: Label Verification and Class Distribution Analysis

In [ ]:
# 1-Check unique labels in train, test, and combined data

train_unique_labels = sorted(train_df["label"].dropna().unique().tolist())
test_unique_labels = sorted(test_df["label"].dropna().unique().tolist())
combined_unique_labels = sorted(combined_eda_df["label"].dropna().unique().tolist())

print("Train unique labels:", train_unique_labels)
print("Test unique labels:", test_unique_labels)
print("Combined unique labels:", combined_unique_labels)

In [ ]:
# 2-Confirm labels are only 0 and 1

allowed_labels = {0, 1}

train_label_check = set(train_unique_labels).issubset(allowed_labels)
test_label_check = set(test_unique_labels).issubset(allowed_labels)
combined_label_check = set(combined_unique_labels).issubset(allowed_labels)

print("Train labels are valid:", train_label_check)
print("Test labels are valid:", test_label_check)
print("Combined labels are valid:", combined_label_check)

if not train_label_check:
    raise ValueError("Train dataset contains labels outside 0 and 1.")

if not test_label_check:
    raise ValueError("Test dataset contains labels outside 0 and 1.")

if not combined_label_check:
    raise ValueError("Combined EDA dataset contains labels outside 0 and 1.")

print("All labels are valid binary labels.")

In [ ]:
# 3-Define label mapping

label_mapping = {
    0: "SAFE",
    1: "INJECTION"
}

label_mapping_df = pd.DataFrame(
    list(label_mapping.items()),
    columns=["label", "label_name"]
)

print("Label mapping defined successfully.")
display(label_mapping_df)

In [ ]:
# 4-Count labels by split

label_count_df = (
    combined_eda_df
    .groupby(["split", "label"])
    .size()
    .reset_index(name="count")
)

label_count_df["label_name"] = label_count_df["label"].map(label_mapping)

print("Label counts by split:")
display(label_count_df)

In [ ]:
# 5-Calculate label percentages by split

label_distribution_df = label_count_df.copy()

label_distribution_df["total_in_split"] = label_distribution_df.groupby("split")["count"].transform("sum")
label_distribution_df["percentage"] = (
    label_distribution_df["count"] / label_distribution_df["total_in_split"] * 100
).round(2)

label_distribution_df = label_distribution_df[
    ["split", "label", "label_name", "count", "total_in_split", "percentage"]
]

print("Label distribution by split:")
display(label_distribution_df)

In [ ]:
# 6-Compare train and test class balance

class_balance_comparison_df = label_distribution_df.pivot(
    index=["label", "label_name"],
    columns="split",
    values="percentage"
).reset_index()

class_balance_comparison_df["percentage_difference_train_minus_test"] = (
    class_balance_comparison_df["train"] - class_balance_comparison_df["test"]
).round(2)

print("Train and test class balance comparison:")
display(class_balance_comparison_df)

In [ ]:
# 7-Save label mapping and label distribution tables

label_mapping_path = TABLES_DIR / "label_mapping.csv"
label_distribution_path = TABLES_DIR / "label_distribution_by_split.csv"
class_balance_path = TABLES_DIR / "class_balance_comparison.csv"

label_mapping_df.to_csv(label_mapping_path, index=False)
label_distribution_df.to_csv(label_distribution_path, index=False)
class_balance_comparison_df.to_csv(class_balance_path, index=False)

print("Label mapping saved to:", label_mapping_path)
print("File exists:", label_mapping_path.exists())

print("\nLabel distribution saved to:", label_distribution_path)
print("File exists:", label_distribution_path.exists())

print("\nClass balance comparison saved to:", class_balance_path)
print("File exists:", class_balance_path.exists())

# Part-5: Label-Based Sample Analysis

In [ ]:
# 1-Display representative SAFE examples

safe_examples_df = (
    combined_eda_df[combined_eda_df["label"] == 0]
    .sample(n=10, random_state=SEED)
    .copy()
)

safe_examples_df["label_name"] = safe_examples_df["label"].map(label_mapping)

safe_examples_df = safe_examples_df[
    ["split", "label", "label_name", "text"]
].reset_index(drop=True)

print("Representative SAFE examples:")
display(safe_examples_df)

In [ ]:
# 2-Display representative INJECTION examples

injection_examples_df = (
    combined_eda_df[combined_eda_df["label"] == 1]
    .sample(n=10, random_state=SEED)
    .copy()
)

injection_examples_df["label_name"] = injection_examples_df["label"].map(label_mapping)

injection_examples_df = injection_examples_df[
    ["split", "label", "label_name", "text"]
].reset_index(drop=True)

print("Representative INJECTION examples:")
display(injection_examples_df)

In [ ]:
# 3-Check whether SAFE examples look like normal user requests

print("Manual inspection checklist for SAFE examples:")
print("1. SAFE examples should look like normal user requests.")
print("2. They should not ask the model to ignore instructions.")
print("3. They should not ask to reveal system, developer, hidden, private, or confidential instructions.")
print("4. They should not contain jailbreak-style behaviour.")

display(safe_examples_df)

In [ ]:
# 4-Check whether INJECTION examples contain adversarial or instruction-hijacking behaviour

print("Manual inspection checklist for INJECTION examples:")
print("1. INJECTION examples may contain instruction override attempts.")
print("2. They may ask the model to ignore previous instructions.")
print("3. They may ask to reveal system prompts, hidden prompts, secrets, or confidential information.")
print("4. They may contain jailbreak-style or role-hijacking behaviour.")

display(injection_examples_df)

In [ ]:
# 5-Save representative examples by label

representative_examples_df = pd.concat(
    [safe_examples_df, injection_examples_df],
    ignore_index=True
)

representative_examples_path = TABLES_DIR / "representative_examples_by_label.csv"

representative_examples_df.to_csv(representative_examples_path, index=False)

print("Representative examples saved to:", representative_examples_path)
print("File exists:", representative_examples_path.exists())

display(representative_examples_df)

#Part-6: Missing, Empty, and Invalid Text Analysis

In [ ]:
# 1-Check missing values in text and label

train_missing_values = train_df[["text", "label"]].isnull().sum()
test_missing_values = test_df[["text", "label"]].isnull().sum()
combined_missing_values = combined_eda_df[["text", "label"]].isnull().sum()

print("Train missing values:")
print(train_missing_values)

print("\nTest missing values:")
print(test_missing_values)

print("\nCombined EDA missing values:")
print(combined_missing_values)

In [ ]:
# 2-Check empty text values

train_empty_text_count = train_df["text"].apply(lambda x: isinstance(x, str) and x == "").sum()
test_empty_text_count = test_df["text"].apply(lambda x: isinstance(x, str) and x == "").sum()
combined_empty_text_count = combined_eda_df["text"].apply(lambda x: isinstance(x, str) and x == "").sum()

print("Train empty text count:", train_empty_text_count)
print("Test empty text count:", test_empty_text_count)
print("Combined EDA empty text count:", combined_empty_text_count)

In [ ]:
# 3-Check whitespace-only text values

train_whitespace_text_count = train_df["text"].apply(lambda x: isinstance(x, str) and x.strip() == "").sum()
test_whitespace_text_count = test_df["text"].apply(lambda x: isinstance(x, str) and x.strip() == "").sum()
combined_whitespace_text_count = combined_eda_df["text"].apply(lambda x: isinstance(x, str) and x.strip() == "").sum()

print("Train whitespace-only text count:", train_whitespace_text_count)
print("Test whitespace-only text count:", test_whitespace_text_count)
print("Combined EDA whitespace-only text count:", combined_whitespace_text_count)

In [ ]:
# 4-Check non-string text values

train_non_string_text_count = train_df["text"].apply(lambda x: not isinstance(x, str)).sum()
test_non_string_text_count = test_df["text"].apply(lambda x: not isinstance(x, str)).sum()
combined_non_string_text_count = combined_eda_df["text"].apply(lambda x: not isinstance(x, str)).sum()

print("Train non-string text count:", train_non_string_text_count)
print("Test non-string text count:", test_non_string_text_count)
print("Combined EDA non-string text count:", combined_non_string_text_count)

In [ ]:
# 5-Check punctuation-only or symbol-only prompts

def is_punctuation_or_symbol_only(text):
    if not isinstance(text, str):
        return False

    stripped_text = text.strip()

    if stripped_text == "":
        return False

    return not any(character.isalnum() for character in stripped_text)

train_symbol_only_count = train_df["text"].apply(is_punctuation_or_symbol_only).sum()
test_symbol_only_count = test_df["text"].apply(is_punctuation_or_symbol_only).sum()
combined_symbol_only_count = combined_eda_df["text"].apply(is_punctuation_or_symbol_only).sum()

symbol_only_examples_df = combined_eda_df[
    combined_eda_df["text"].apply(is_punctuation_or_symbol_only)
][["split", "label", "text"]].copy()

print("Train punctuation/symbol-only text count:", train_symbol_only_count)
print("Test punctuation/symbol-only text count:", test_symbol_only_count)
print("Combined EDA punctuation/symbol-only text count:", combined_symbol_only_count)

print("\nPunctuation/symbol-only examples:")
display(symbol_only_examples_df)

In [ ]:
# 6-Check very short and long prompts

length_check_df = combined_eda_df[["split", "label", "text"]].copy()

length_check_df["character_length"] = length_check_df["text"].apply(lambda x: len(x) if isinstance(x, str) else 0)
length_check_df["word_count"] = length_check_df["text"].apply(lambda x: len(x.split()) if isinstance(x, str) else 0)

very_short_prompts_df = length_check_df[
    (length_check_df["character_length"] <= 10) | (length_check_df["word_count"] <= 2)
].copy()

longest_prompts_df = length_check_df.sort_values(
    by="character_length",
    ascending=False
).head(10).copy()

print("Very short prompt count:", len(very_short_prompts_df))
print("Longest prompt examples selected:", len(longest_prompts_df))

print("\nVery short prompt examples:")
display(very_short_prompts_df)

print("\nTop 10 longest prompt examples:")
display(longest_prompts_df)

In [ ]:
# 7-Save missing empty and invalid text analysis report

missing_invalid_report = {
    "train_missing_text": train_df["text"].isnull().sum(),
    "train_missing_label": train_df["label"].isnull().sum(),
    "test_missing_text": test_df["text"].isnull().sum(),
    "test_missing_label": test_df["label"].isnull().sum(),
    "combined_missing_text": combined_eda_df["text"].isnull().sum(),
    "combined_missing_label": combined_eda_df["label"].isnull().sum(),
    "train_empty_text_count": train_df["text"].apply(lambda x: isinstance(x, str) and x == "").sum(),
    "test_empty_text_count": test_df["text"].apply(lambda x: isinstance(x, str) and x == "").sum(),
    "combined_empty_text_count": combined_eda_df["text"].apply(lambda x: isinstance(x, str) and x == "").sum(),
    "train_whitespace_only_text_count": train_df["text"].apply(lambda x: isinstance(x, str) and x.strip() == "").sum(),
    "test_whitespace_only_text_count": test_df["text"].apply(lambda x: isinstance(x, str) and x.strip() == "").sum(),
    "combined_whitespace_only_text_count": combined_eda_df["text"].apply(lambda x: isinstance(x, str) and x.strip() == "").sum(),
    "combined_very_short_prompt_count": len(very_short_prompts_df),
    "longest_prompt_examples_selected": len(longest_prompts_df)
}

missing_invalid_report_df = pd.DataFrame(
    list(missing_invalid_report.items()),
    columns=["check", "count"]
)

missing_invalid_report_path = TABLES_DIR / "missing_empty_invalid_text_report.csv"
very_short_prompts_path = TABLES_DIR / "very_short_prompt_examples.csv"
longest_prompts_path = TABLES_DIR / "longest_prompt_examples.csv"

missing_invalid_report_df.to_csv(missing_invalid_report_path, index=False)
very_short_prompts_df.to_csv(very_short_prompts_path, index=False)
longest_prompts_df.to_csv(longest_prompts_path, index=False)

print("Missing/empty/invalid text report saved to:", missing_invalid_report_path)
print("File exists:", missing_invalid_report_path.exists())

print("\nVery short prompt examples saved to:", very_short_prompts_path)
print("File exists:", very_short_prompts_path.exists())

print("\nLongest prompt examples saved to:", longest_prompts_path)
print("File exists:", longest_prompts_path.exists())

display(missing_invalid_report_df)

#Part-7 : Hidden, Control, and Unusual Character Analysis

In [ ]:
# 1-Check newline characters

train_newline_count = train_df["text"].apply(lambda x: isinstance(x, str) and "\n" in x).sum()
test_newline_count = test_df["text"].apply(lambda x: isinstance(x, str) and "\n" in x).sum()
combined_newline_count = combined_eda_df["text"].apply(lambda x: isinstance(x, str) and "\n" in x).sum()

newline_examples_df = combined_eda_df[
    combined_eda_df["text"].apply(lambda x: isinstance(x, str) and "\n" in x)
][["split", "label", "text"]].copy()

print("Train newline character count:", train_newline_count)
print("Test newline character count:", test_newline_count)
print("Combined EDA newline character count:", combined_newline_count)

print("\nNewline character examples:")
display(newline_examples_df)

In [ ]:
# 2-Check tab characters

train_tab_count = train_df["text"].apply(lambda x: isinstance(x, str) and "\t" in x).sum()
test_tab_count = test_df["text"].apply(lambda x: isinstance(x, str) and "\t" in x).sum()
combined_tab_count = combined_eda_df["text"].apply(lambda x: isinstance(x, str) and "\t" in x).sum()

tab_examples_df = combined_eda_df[
    combined_eda_df["text"].apply(lambda x: isinstance(x, str) and "\t" in x)
][["split", "label", "text"]].copy()

print("Train tab character count:", train_tab_count)
print("Test tab character count:", test_tab_count)
print("Combined EDA tab character count:", combined_tab_count)

print("\nTab character examples:")
display(tab_examples_df)

In [ ]:
# 3-Check carriage-return characters

train_carriage_return_count = train_df["text"].apply(lambda x: isinstance(x, str) and "\r" in x).sum()
test_carriage_return_count = test_df["text"].apply(lambda x: isinstance(x, str) and "\r" in x).sum()
combined_carriage_return_count = combined_eda_df["text"].apply(lambda x: isinstance(x, str) and "\r" in x).sum()

carriage_return_examples_df = combined_eda_df[
    combined_eda_df["text"].apply(lambda x: isinstance(x, str) and "\r" in x)
][["split", "label", "text"]].copy()

print("Train carriage-return character count:", train_carriage_return_count)
print("Test carriage-return character count:", test_carriage_return_count)
print("Combined EDA carriage-return character count:", combined_carriage_return_count)

print("\nCarriage-return character examples:")
display(carriage_return_examples_df)

In [ ]:
# 4-Check repeated spaces

train_repeated_space_count = train_df["text"].apply(lambda x: isinstance(x, str) and "  " in x).sum()
test_repeated_space_count = test_df["text"].apply(lambda x: isinstance(x, str) and "  " in x).sum()
combined_repeated_space_count = combined_eda_df["text"].apply(lambda x: isinstance(x, str) and "  " in x).sum()

repeated_space_examples_df = combined_eda_df[
    combined_eda_df["text"].apply(lambda x: isinstance(x, str) and "  " in x)
][["split", "label", "text"]].copy()

print("Train repeated-space text count:", train_repeated_space_count)
print("Test repeated-space text count:", test_repeated_space_count)
print("Combined EDA repeated-space text count:", combined_repeated_space_count)

print("\nRepeated-space examples:")
display(repeated_space_examples_df)

In [ ]:
# 5-Check other non-printable control characters

def has_other_control_character(text):
    if not isinstance(text, str):
        return False

    allowed_control_characters = {"\n", "\t", "\r"}

    for character in text:
        if character in allowed_control_characters:
            continue

        if ord(character) < 32 or ord(character) == 127:
            return True

    return False

train_other_control_count = train_df["text"].apply(has_other_control_character).sum()
test_other_control_count = test_df["text"].apply(has_other_control_character).sum()
combined_other_control_count = combined_eda_df["text"].apply(has_other_control_character).sum()

other_control_examples_df = combined_eda_df[
    combined_eda_df["text"].apply(has_other_control_character)
][["split", "label", "text"]].copy()

print("Train other control-character text count:", train_other_control_count)
print("Test other control-character text count:", test_other_control_count)
print("Combined EDA other control-character text count:", combined_other_control_count)

print("\nOther control-character examples:")
display(other_control_examples_df)

In [ ]:
# 6-Check zero-width and invisible characters

zero_width_characters = {
    "\u200b": "zero_width_space",
    "\u200c": "zero_width_non_joiner",
    "\u200d": "zero_width_joiner",
    "\ufeff": "byte_order_mark",
    "\u2060": "word_joiner",
    "\u00ad": "soft_hyphen"
}

def has_zero_width_character(text):
    if not isinstance(text, str):
        return False

    return any(character in text for character in zero_width_characters.keys())

train_zero_width_count = train_df["text"].apply(has_zero_width_character).sum()
test_zero_width_count = test_df["text"].apply(has_zero_width_character).sum()
combined_zero_width_count = combined_eda_df["text"].apply(has_zero_width_character).sum()

zero_width_examples_df = combined_eda_df[
    combined_eda_df["text"].apply(has_zero_width_character)
][["split", "label", "text"]].copy()

print("Train zero-width/invisible character text count:", train_zero_width_count)
print("Test zero-width/invisible character text count:", test_zero_width_count)
print("Combined EDA zero-width/invisible character text count:", combined_zero_width_count)

print("\nZero-width/invisible character examples:")
display(zero_width_examples_df)

In [ ]:
# 7-Count prompts containing unusual characters

def has_newline_character(text):
    return isinstance(text, str) and "\n" in text

def has_tab_character(text):
    return isinstance(text, str) and "\t" in text

def has_carriage_return_character(text):
    return isinstance(text, str) and "\r" in text

def has_repeated_space(text):
    return isinstance(text, str) and "  " in text

unusual_character_analysis_df = combined_eda_df[["split", "label", "text"]].copy()

unusual_character_analysis_df["has_newline"] = unusual_character_analysis_df["text"].apply(has_newline_character)
unusual_character_analysis_df["has_tab"] = unusual_character_analysis_df["text"].apply(has_tab_character)
unusual_character_analysis_df["has_carriage_return"] = unusual_character_analysis_df["text"].apply(has_carriage_return_character)
unusual_character_analysis_df["has_repeated_space"] = unusual_character_analysis_df["text"].apply(has_repeated_space)
unusual_character_analysis_df["has_other_control_character"] = unusual_character_analysis_df["text"].apply(has_other_control_character)
unusual_character_analysis_df["has_zero_width_character"] = unusual_character_analysis_df["text"].apply(has_zero_width_character)

unusual_character_columns = [
    "has_newline",
    "has_tab",
    "has_carriage_return",
    "has_repeated_space",
    "has_other_control_character",
    "has_zero_width_character"
]

unusual_character_analysis_df["has_any_unusual_character"] = unusual_character_analysis_df[
    unusual_character_columns
].any(axis=1)

unusual_character_count_by_split_df = (
    unusual_character_analysis_df
    .groupby("split")["has_any_unusual_character"]
    .sum()
    .reset_index(name="unusual_character_prompt_count")
)

unusual_character_count_by_label_df = (
    unusual_character_analysis_df
    .groupby("label")["has_any_unusual_character"]
    .sum()
    .reset_index(name="unusual_character_prompt_count")
)

unusual_character_examples_df = unusual_character_analysis_df[
    unusual_character_analysis_df["has_any_unusual_character"]
].copy()

print("Total prompts containing any unusual character:", unusual_character_analysis_df["has_any_unusual_character"].sum())

print("\nUnusual-character prompt count by split:")
display(unusual_character_count_by_split_df)

print("\nUnusual-character prompt count by label:")
display(unusual_character_count_by_label_df)

print("\nUnusual-character examples:")
display(unusual_character_examples_df)

In [ ]:
# 8-Save unusual-character summary and examples

unusual_character_summary = {
    "train_newline_count": train_newline_count,
    "test_newline_count": test_newline_count,
    "combined_newline_count": combined_newline_count,
    "train_tab_count": train_tab_count,
    "test_tab_count": test_tab_count,
    "combined_tab_count": combined_tab_count,
    "train_carriage_return_count": train_carriage_return_count,
    "test_carriage_return_count": test_carriage_return_count,
    "combined_carriage_return_count": combined_carriage_return_count,
    "train_repeated_space_count": train_repeated_space_count,
    "test_repeated_space_count": test_repeated_space_count,
    "combined_repeated_space_count": combined_repeated_space_count,
    "train_other_control_count": train_other_control_count,
    "test_other_control_count": test_other_control_count,
    "combined_other_control_count": combined_other_control_count,
    "train_zero_width_count": train_zero_width_count,
    "test_zero_width_count": test_zero_width_count,
    "combined_zero_width_count": combined_zero_width_count,
    "total_prompts_with_any_unusual_character": unusual_character_analysis_df["has_any_unusual_character"].sum()
}

unusual_character_summary_df = pd.DataFrame(
    list(unusual_character_summary.items()),
    columns=["check", "count"]
)

unusual_character_summary_path = TABLES_DIR / "unusual_character_summary.csv"
unusual_character_by_split_path = TABLES_DIR / "unusual_character_count_by_split.csv"
unusual_character_by_label_path = TABLES_DIR / "unusual_character_count_by_label.csv"
unusual_character_examples_path = TABLES_DIR / "unusual_character_examples.csv"

unusual_character_summary_df.to_csv(unusual_character_summary_path, index=False)
unusual_character_count_by_split_df.to_csv(unusual_character_by_split_path, index=False)
unusual_character_count_by_label_df.to_csv(unusual_character_by_label_path, index=False)
unusual_character_examples_df.to_csv(unusual_character_examples_path, index=False)

print("Unusual-character summary saved to:", unusual_character_summary_path)
print("File exists:", unusual_character_summary_path.exists())

print("\nUnusual-character count by split saved to:", unusual_character_by_split_path)
print("File exists:", unusual_character_by_split_path.exists())

print("\nUnusual-character count by label saved to:", unusual_character_by_label_path)
print("File exists:", unusual_character_by_label_path.exists())

print("\nUnusual-character examples saved to:", unusual_character_examples_path)
print("File exists:", unusual_character_examples_path.exists())

display(unusual_character_summary_df)

#Part-8: Duplicate and Train/Test Leakage Analysis

In [ ]:
# 1-Check duplicate full rows

train_duplicate_full_row_count = train_df.duplicated().sum()
test_duplicate_full_row_count = test_df.duplicated().sum()
combined_duplicate_full_row_count = combined_eda_df.duplicated(subset=["text", "label", "split"]).sum()

duplicate_full_rows_df = combined_eda_df[
    combined_eda_df.duplicated(subset=["text", "label", "split"], keep=False)
].copy()

print("Train duplicate full-row count:", train_duplicate_full_row_count)
print("Test duplicate full-row count:", test_duplicate_full_row_count)
print("Combined EDA duplicate full-row count:", combined_duplicate_full_row_count)

print("\nDuplicate full-row examples:")
display(duplicate_full_rows_df)

In [ ]:
# 2-Check duplicate text values

train_duplicate_text_count = train_df.duplicated(subset=["text"]).sum()
test_duplicate_text_count = test_df.duplicated(subset=["text"]).sum()
combined_duplicate_text_count = combined_eda_df.duplicated(subset=["text"]).sum()

duplicate_text_examples_df = combined_eda_df[
    combined_eda_df.duplicated(subset=["text"], keep=False)
].copy()

print("Train duplicate text count:", train_duplicate_text_count)
print("Test duplicate text count:", test_duplicate_text_count)
print("Combined EDA duplicate text count:", combined_duplicate_text_count)

print("\nDuplicate text examples:")
display(duplicate_text_examples_df)

In [ ]:
# 3-Check duplicate texts with same labels

train_duplicate_text_same_label_count = train_df.duplicated(subset=["text", "label"]).sum()
test_duplicate_text_same_label_count = test_df.duplicated(subset=["text", "label"]).sum()
combined_duplicate_text_same_label_count = combined_eda_df.duplicated(subset=["text", "label"]).sum()

duplicate_text_same_label_examples_df = combined_eda_df[
    combined_eda_df.duplicated(subset=["text", "label"], keep=False)
].copy()

print("Train duplicate text with same label count:", train_duplicate_text_same_label_count)
print("Test duplicate text with same label count:", test_duplicate_text_same_label_count)
print("Combined EDA duplicate text with same label count:", combined_duplicate_text_same_label_count)

print("\nDuplicate text with same label examples:")
display(duplicate_text_same_label_examples_df)

In [ ]:
# 4-Check duplicate texts with conflicting labels

text_label_counts_df = (
    combined_eda_df
    .groupby("text")["label"]
    .nunique()
    .reset_index(name="unique_label_count")
)

conflicting_texts = text_label_counts_df[
    text_label_counts_df["unique_label_count"] > 1
]["text"]

conflicting_label_examples_df = combined_eda_df[
    combined_eda_df["text"].isin(conflicting_texts)
].copy()

conflicting_label_text_count = len(conflicting_texts)
conflicting_label_row_count = len(conflicting_label_examples_df)

print("Texts with conflicting labels count:", conflicting_label_text_count)
print("Rows involved in conflicting labels:", conflicting_label_row_count)

print("\nConflicting label examples:")
display(conflicting_label_examples_df)

In [ ]:
# 4-Check duplicate texts with conflicting labels

text_label_counts_df = (
    combined_eda_df
    .groupby("text")["label"]
    .nunique()
    .reset_index(name="unique_label_count")
)

conflicting_texts = text_label_counts_df[
    text_label_counts_df["unique_label_count"] > 1
]["text"]

conflicting_label_examples_df = combined_eda_df[
    combined_eda_df["text"].isin(conflicting_texts)
].copy()

conflicting_label_text_count = len(conflicting_texts)
conflicting_label_row_count = len(conflicting_label_examples_df)

print("Texts with conflicting labels count:", conflicting_label_text_count)
print("Rows involved in conflicting labels:", conflicting_label_row_count)

print("\nConflicting label examples:")
display(conflicting_label_examples_df)

In [ ]:
# 5-Check exact train test text overlap

train_text_set = set(train_df["text"])
test_text_set = set(test_df["text"])

overlap_texts = train_text_set.intersection(test_text_set)

exact_train_test_overlap_df = combined_eda_df[
    combined_eda_df["text"].isin(overlap_texts)
].copy()

print("Exact train/test overlapping text count:", len(overlap_texts))
print("Rows involved in exact train/test overlap:", len(exact_train_test_overlap_df))

print("\nExact train/test overlap examples:")
display(exact_train_test_overlap_df)

In [ ]:
# 6-Create temporary normalised text for analysis only

duplicate_leakage_analysis_df = combined_eda_df[["split", "label", "text"]].copy()

def normalise_text_for_analysis(text):
    if not isinstance(text, str):
        return ""

    text = text.lower()
    text = text.strip()
    text = re.sub(r"\s+", " ", text)

    return text

duplicate_leakage_analysis_df["normalised_text"] = duplicate_leakage_analysis_df["text"].apply(normalise_text_for_analysis)

print("Temporary normalised text created for analysis only.")
print("Original combined EDA shape:", combined_eda_df.shape)
print("Duplicate/leakage analysis shape:", duplicate_leakage_analysis_df.shape)

print("\nColumns:")
print(duplicate_leakage_analysis_df.columns.tolist())

print("\nFirst 5 rows:")
display(duplicate_leakage_analysis_df.head())

In [ ]:
# 7-Check normalised duplicate text values

normalised_duplicate_text_count = duplicate_leakage_analysis_df.duplicated(
    subset=["normalised_text"]
).sum()

normalised_duplicate_examples_df = duplicate_leakage_analysis_df[
    duplicate_leakage_analysis_df.duplicated(
        subset=["normalised_text"],
        keep=False
    )
].copy()

print("Normalised duplicate text count:", normalised_duplicate_text_count)

print("\nNormalised duplicate text examples:")
display(normalised_duplicate_examples_df)

In [ ]:
# 8-Check normalised train test overlap

normalised_train_text_set = set(
    duplicate_leakage_analysis_df[
        duplicate_leakage_analysis_df["split"] == "train"
    ]["normalised_text"]
)

normalised_test_text_set = set(
    duplicate_leakage_analysis_df[
        duplicate_leakage_analysis_df["split"] == "test"
    ]["normalised_text"]
)

normalised_overlap_texts = normalised_train_text_set.intersection(normalised_test_text_set)

normalised_train_test_overlap_df = duplicate_leakage_analysis_df[
    duplicate_leakage_analysis_df["normalised_text"].isin(normalised_overlap_texts)
].copy()

print("Normalised train/test overlapping text count:", len(normalised_overlap_texts))
print("Rows involved in normalised train/test overlap:", len(normalised_train_test_overlap_df))

print("\nNormalised train/test overlap examples:")
display(normalised_train_test_overlap_df)

In [ ]:
# 9-Check conflicting labels after normalisation

normalised_text_label_counts_df = (
    duplicate_leakage_analysis_df
    .groupby("normalised_text")["label"]
    .nunique()
    .reset_index(name="unique_label_count")
)

normalised_conflicting_texts = normalised_text_label_counts_df[
    normalised_text_label_counts_df["unique_label_count"] > 1
]["normalised_text"]

normalised_conflicting_label_examples_df = duplicate_leakage_analysis_df[
    duplicate_leakage_analysis_df["normalised_text"].isin(normalised_conflicting_texts)
].copy()

normalised_conflicting_label_text_count = len(normalised_conflicting_texts)
normalised_conflicting_label_row_count = len(normalised_conflicting_label_examples_df)

print("Normalised texts with conflicting labels count:", normalised_conflicting_label_text_count)
print("Rows involved in normalised conflicting labels:", normalised_conflicting_label_row_count)

print("\nNormalised conflicting label examples:")
display(normalised_conflicting_label_examples_df)

In [ ]:
# 10-Save duplicate and leakage reports

duplicate_leakage_summary = {
    "train_duplicate_full_row_count": train_duplicate_full_row_count,
    "test_duplicate_full_row_count": test_duplicate_full_row_count,
    "combined_duplicate_full_row_count": combined_duplicate_full_row_count,
    "train_duplicate_text_count": train_duplicate_text_count,
    "test_duplicate_text_count": test_duplicate_text_count,
    "combined_duplicate_text_count": combined_duplicate_text_count,
    "train_duplicate_text_same_label_count": train_duplicate_text_same_label_count,
    "test_duplicate_text_same_label_count": test_duplicate_text_same_label_count,
    "combined_duplicate_text_same_label_count": combined_duplicate_text_same_label_count,
    "conflicting_label_text_count": conflicting_label_text_count,
    "conflicting_label_row_count": conflicting_label_row_count,
    "exact_train_test_overlapping_text_count": len(overlap_texts),
    "exact_train_test_overlap_row_count": len(exact_train_test_overlap_df),
    "normalised_duplicate_text_count": normalised_duplicate_text_count,
    "normalised_train_test_overlapping_text_count": len(normalised_overlap_texts),
    "normalised_train_test_overlap_row_count": len(normalised_train_test_overlap_df),
    "normalised_conflicting_label_text_count": normalised_conflicting_label_text_count,
    "normalised_conflicting_label_row_count": normalised_conflicting_label_row_count
}

duplicate_leakage_summary_df = pd.DataFrame(
    list(duplicate_leakage_summary.items()),
    columns=["check", "count"]
)

duplicate_leakage_summary_path = TABLES_DIR / "duplicate_leakage_summary.csv"
duplicate_full_rows_path = TABLES_DIR / "duplicate_full_row_examples.csv"
duplicate_text_examples_path = TABLES_DIR / "duplicate_text_examples.csv"
conflicting_label_examples_path = TABLES_DIR / "conflicting_label_examples.csv"
exact_train_test_overlap_path = TABLES_DIR / "exact_train_test_overlap_examples.csv"
normalised_duplicate_examples_path = TABLES_DIR / "normalised_duplicate_text_examples.csv"
normalised_train_test_overlap_path = TABLES_DIR / "normalised_train_test_overlap_examples.csv"
normalised_conflicting_label_examples_path = TABLES_DIR / "normalised_conflicting_label_examples.csv"

duplicate_leakage_summary_df.to_csv(duplicate_leakage_summary_path, index=False)
duplicate_full_rows_df.to_csv(duplicate_full_rows_path, index=False)
duplicate_text_examples_df.to_csv(duplicate_text_examples_path, index=False)
conflicting_label_examples_df.to_csv(conflicting_label_examples_path, index=False)
exact_train_test_overlap_df.to_csv(exact_train_test_overlap_path, index=False)
normalised_duplicate_examples_df.to_csv(normalised_duplicate_examples_path, index=False)
normalised_train_test_overlap_df.to_csv(normalised_train_test_overlap_path, index=False)
normalised_conflicting_label_examples_df.to_csv(normalised_conflicting_label_examples_path, index=False)

print("Duplicate/leakage summary saved to:", duplicate_leakage_summary_path)
print("File exists:", duplicate_leakage_summary_path.exists())

print("\nDuplicate full-row examples saved to:", duplicate_full_rows_path)
print("File exists:", duplicate_full_rows_path.exists())

print("\nDuplicate text examples saved to:", duplicate_text_examples_path)
print("File exists:", duplicate_text_examples_path.exists())

print("\nConflicting label examples saved to:", conflicting_label_examples_path)
print("File exists:", conflicting_label_examples_path.exists())

print("\nExact train/test overlap examples saved to:", exact_train_test_overlap_path)
print("File exists:", exact_train_test_overlap_path.exists())

print("\nNormalised duplicate text examples saved to:", normalised_duplicate_examples_path)
print("File exists:", normalised_duplicate_examples_path.exists())

print("\nNormalised train/test overlap examples saved to:", normalised_train_test_overlap_path)
print("File exists:", normalised_train_test_overlap_path.exists())

print("\nNormalised conflicting label examples saved to:", normalised_conflicting_label_examples_path)
print("File exists:", normalised_conflicting_label_examples_path.exists())

display(duplicate_leakage_summary_df)

#Part-9: Text Length and Transformer Token-Length Analysis.

In [ ]:
# 1-Add temporary character-length column

text_length_analysis_df = combined_eda_df[["split", "label", "text"]].copy()

text_length_analysis_df["character_length"] = text_length_analysis_df["text"].apply(
    lambda x: len(x) if isinstance(x, str) else 0
)

print("Temporary character-length column added.")
print("Text length analysis shape:", text_length_analysis_df.shape)

print("\nColumns:")
print(text_length_analysis_df.columns.tolist())

print("\nFirst 5 rows:")
display(text_length_analysis_df.head())

In [ ]:
# 2-Add temporary word-count column

text_length_analysis_df["word_count"] = text_length_analysis_df["text"].apply(
    lambda x: len(x.split()) if isinstance(x, str) else 0
)

print("Temporary word-count column added.")
print("Text length analysis shape:", text_length_analysis_df.shape)

print("\nColumns:")
print(text_length_analysis_df.columns.tolist())

print("\nFirst 5 rows:")
display(text_length_analysis_df.head())

In [ ]:
# 3-Calculate character-length summary

character_length_summary_by_split_label_df = (
    text_length_analysis_df
    .groupby(["split", "label"])
    .agg(
        row_count=("text", "count"),
        min_character_length=("character_length", "min"),
        median_character_length=("character_length", "median"),
        mean_character_length=("character_length", "mean"),
        max_character_length=("character_length", "max")
    )
    .reset_index()
)

character_length_summary_by_split_label_df["label_name"] = (
    character_length_summary_by_split_label_df["label"].map(label_mapping)
)

character_length_summary_by_split_label_df = character_length_summary_by_split_label_df[
    [
        "split",
        "label",
        "label_name",
        "row_count",
        "min_character_length",
        "median_character_length",
        "mean_character_length",
        "max_character_length"
    ]
]

character_length_summary_by_split_label_df[
    ["median_character_length", "mean_character_length"]
] = character_length_summary_by_split_label_df[
    ["median_character_length", "mean_character_length"]
].round(2)

print("Character-length summary by split and label:")
display(character_length_summary_by_split_label_df)

In [ ]:
# 4-Calculate word-count summary

word_count_summary_by_split_label_df = (
    text_length_analysis_df
    .groupby(["split", "label"])
    .agg(
        row_count=("text", "count"),
        min_word_count=("word_count", "min"),
        median_word_count=("word_count", "median"),
        mean_word_count=("word_count", "mean"),
        max_word_count=("word_count", "max")
    )
    .reset_index()
)

word_count_summary_by_split_label_df["label_name"] = (
    word_count_summary_by_split_label_df["label"].map(label_mapping)
)

word_count_summary_by_split_label_df = word_count_summary_by_split_label_df[
    [
        "split",
        "label",
        "label_name",
        "row_count",
        "min_word_count",
        "median_word_count",
        "mean_word_count",
        "max_word_count"
    ]
]

word_count_summary_by_split_label_df[
    ["median_word_count", "mean_word_count"]
] = word_count_summary_by_split_label_df[
    ["median_word_count", "mean_word_count"]
].round(2)

print("Word-count summary by split and label:")
display(word_count_summary_by_split_label_df)

In [ ]:
# 5-Inspect shortest prompts

shortest_prompts_df = text_length_analysis_df.copy()

shortest_prompts_df["label_name"] = shortest_prompts_df["label"].map(label_mapping)

shortest_prompts_df = shortest_prompts_df.sort_values(
    by=["word_count", "character_length"],
    ascending=[True, True]
).head(15)

print("Shortest prompts by word count:")
display(
    shortest_prompts_df[
        [
            "split",
            "label",
            "label_name",
            "text",
            "character_length",
            "word_count"
        ]
    ]
)

In [ ]:
# 6-Inspect longest prompts

longest_prompts_df = text_length_analysis_df.copy()

longest_prompts_df["label_name"] = longest_prompts_df["label"].map(label_mapping)

longest_prompts_df = longest_prompts_df.sort_values(
    by=["word_count", "character_length"],
    ascending=[False, False]
).head(15)

print("Longest prompts by word count:")
display(
    longest_prompts_df[
        [
            "split",
            "label",
            "label_name",
            "text",
            "character_length",
            "word_count"
        ]
    ]
)

In [ ]:
# 7-Load tokenizers for the planned transformer models

try:
    from transformers import AutoTokenizer
except ImportError:
    !pip -q install transformers
    from transformers import AutoTokenizer

model_checkpoints = {
    "distilbert": "distilbert-base-uncased",
    "roberta": "roberta-base",
    "secbert": "jackaduma/SecBERT"
}

tokenizers_for_analysis = {}

for model_name, checkpoint in model_checkpoints.items():
    try:
        tokenizer = AutoTokenizer.from_pretrained(checkpoint, use_fast=True)
    except Exception:
        tokenizer = AutoTokenizer.from_pretrained(checkpoint, use_fast=False)

    tokenizers_for_analysis[model_name] = tokenizer

    print(model_name.upper(), "tokenizer loaded.")
    print("Checkpoint:", checkpoint)
    print("Tokenizer class:", tokenizer.__class__.__name__)
    print("Model max length:", tokenizer.model_max_length)
    print("-" * 70)

distilbert_tokenizer = tokenizers_for_analysis["distilbert"]
roberta_tokenizer = tokenizers_for_analysis["roberta"]
secbert_tokenizer = tokenizers_for_analysis["secbert"]

print("All planned model tokenizers loaded successfully.")

In [ ]:
# 8-Calculate token lengths for DistilBERT

def calculate_token_length(text, tokenizer):
    if not isinstance(text, str):
        return 0

    return len(
        tokenizer.encode(
            text,
            add_special_tokens=True,
            truncation=False
        )
    )

text_length_analysis_df["distilbert_token_length"] = text_length_analysis_df["text"].apply(
    lambda x: calculate_token_length(x, distilbert_tokenizer)
)

distilbert_token_summary_df = (
    text_length_analysis_df
    .groupby(["split", "label"])
    .agg(
        row_count=("text", "count"),
        min_distilbert_tokens=("distilbert_token_length", "min"),
        median_distilbert_tokens=("distilbert_token_length", "median"),
        mean_distilbert_tokens=("distilbert_token_length", "mean"),
        max_distilbert_tokens=("distilbert_token_length", "max")
    )
    .reset_index()
)

distilbert_token_summary_df["label_name"] = distilbert_token_summary_df["label"].map(label_mapping)

distilbert_token_summary_df = distilbert_token_summary_df[
    [
        "split",
        "label",
        "label_name",
        "row_count",
        "min_distilbert_tokens",
        "median_distilbert_tokens",
        "mean_distilbert_tokens",
        "max_distilbert_tokens"
    ]
]

distilbert_token_summary_df[
    ["median_distilbert_tokens", "mean_distilbert_tokens"]
] = distilbert_token_summary_df[
    ["median_distilbert_tokens", "mean_distilbert_tokens"]
].round(2)

print("DistilBERT token-length summary by split and label:")
display(distilbert_token_summary_df)

print("\nFirst 5 rows with DistilBERT token length:")
display(
    text_length_analysis_df[
        [
            "split",
            "label",
            "text",
            "character_length",
            "word_count",
            "distilbert_token_length"
        ]
    ].head()
)

In [ ]:
# 9-Calculate token lengths for RoBERTa

text_length_analysis_df["roberta_token_length"] = text_length_analysis_df["text"].apply(
    lambda x: calculate_token_length(x, roberta_tokenizer)
)

roberta_token_summary_df = (
    text_length_analysis_df
    .groupby(["split", "label"])
    .agg(
        row_count=("text", "count"),
        min_roberta_tokens=("roberta_token_length", "min"),
        median_roberta_tokens=("roberta_token_length", "median"),
        mean_roberta_tokens=("roberta_token_length", "mean"),
        max_roberta_tokens=("roberta_token_length", "max")
    )
    .reset_index()
)

roberta_token_summary_df["label_name"] = roberta_token_summary_df["label"].map(label_mapping)

roberta_token_summary_df = roberta_token_summary_df[
    [
        "split",
        "label",
        "label_name",
        "row_count",
        "min_roberta_tokens",
        "median_roberta_tokens",
        "mean_roberta_tokens",
        "max_roberta_tokens"
    ]
]

roberta_token_summary_df[
    ["median_roberta_tokens", "mean_roberta_tokens"]
] = roberta_token_summary_df[
    ["median_roberta_tokens", "mean_roberta_tokens"]
].round(2)

print("RoBERTa token-length summary by split and label:")
display(roberta_token_summary_df)

print("\nFirst 5 rows with RoBERTa token length:")
display(
    text_length_analysis_df[
        [
            "split",
            "label",
            "text",
            "character_length",
            "word_count",
            "distilbert_token_length",
            "roberta_token_length"
        ]
    ].head()
)

In [ ]:
# 10-Calculate token lengths for SecBERT after confirming the final SecBERT checkpoint

final_secbert_checkpoint = model_checkpoints["secbert"]

print("Final SecBERT checkpoint confirmed:", final_secbert_checkpoint)

text_length_analysis_df["secbert_token_length"] = text_length_analysis_df["text"].apply(
    lambda x: calculate_token_length(x, secbert_tokenizer)
)

secbert_token_summary_df = (
    text_length_analysis_df
    .groupby(["split", "label"])
    .agg(
        row_count=("text", "count"),
        min_secbert_tokens=("secbert_token_length", "min"),
        median_secbert_tokens=("secbert_token_length", "median"),
        mean_secbert_tokens=("secbert_token_length", "mean"),
        max_secbert_tokens=("secbert_token_length", "max")
    )
    .reset_index()
)

secbert_token_summary_df["label_name"] = secbert_token_summary_df["label"].map(label_mapping)

secbert_token_summary_df = secbert_token_summary_df[
    [
        "split",
        "label",
        "label_name",
        "row_count",
        "min_secbert_tokens",
        "median_secbert_tokens",
        "mean_secbert_tokens",
        "max_secbert_tokens"
    ]
]

secbert_token_summary_df[
    ["median_secbert_tokens", "mean_secbert_tokens"]
] = secbert_token_summary_df[
    ["median_secbert_tokens", "mean_secbert_tokens"]
].round(2)

print("\nSecBERT token-length summary by split and label:")
display(secbert_token_summary_df)

print("\nFirst 5 rows with SecBERT token length:")
display(
    text_length_analysis_df[
        [
            "split",
            "label",
            "text",
            "character_length",
            "word_count",
            "distilbert_token_length",
            "roberta_token_length",
            "secbert_token_length"
        ]
    ].head()
)

In [ ]:
# 11-Calculate percentage of prompts longer than 64, 128, 256, and 512 tokens

token_thresholds = [64, 128, 256, 512]

token_length_columns = {
    "distilbert": "distilbert_token_length",
    "roberta": "roberta_token_length",
    "secbert": "secbert_token_length"
}

overall_token_threshold_rows = []

for model_name, token_column in token_length_columns.items():
    total_rows = len(text_length_analysis_df)

    for threshold in token_thresholds:
        longer_count = (text_length_analysis_df[token_column] > threshold).sum()
        longer_percentage = (longer_count / total_rows) * 100

        overall_token_threshold_rows.append(
            {
                "model": model_name,
                "token_length_column": token_column,
                "threshold": threshold,
                "total_rows": total_rows,
                "longer_than_threshold_count": longer_count,
                "longer_than_threshold_percentage": round(longer_percentage, 2)
            }
        )

overall_token_threshold_summary_df = pd.DataFrame(overall_token_threshold_rows)

split_label_token_threshold_rows = []

for model_name, token_column in token_length_columns.items():
    for (split_name, label_value), group_df in text_length_analysis_df.groupby(["split", "label"]):
        total_rows = len(group_df)

        for threshold in token_thresholds:
            longer_count = (group_df[token_column] > threshold).sum()
            longer_percentage = (longer_count / total_rows) * 100

            split_label_token_threshold_rows.append(
                {
                    "model": model_name,
                    "token_length_column": token_column,
                    "split": split_name,
                    "label": label_value,
                    "label_name": label_mapping[label_value],
                    "threshold": threshold,
                    "total_rows": total_rows,
                    "longer_than_threshold_count": longer_count,
                    "longer_than_threshold_percentage": round(longer_percentage, 2)
                }
            )

token_threshold_summary_by_split_label_df = pd.DataFrame(split_label_token_threshold_rows)

print("Overall token threshold summary:")
display(overall_token_threshold_summary_df)

print("\nToken threshold summary by split and label:")
display(token_threshold_summary_by_split_label_df)

In [ ]:
# 12-Save character, word, and token-length summaries

text_length_analysis_output_path = TABLES_DIR / "text_length_analysis_with_token_lengths.csv"

character_length_summary_path = TABLES_DIR / "character_length_summary_by_split_label.csv"
word_count_summary_path = TABLES_DIR / "word_count_summary_by_split_label.csv"

shortest_prompts_path = TABLES_DIR / "shortest_prompts_by_word_count.csv"
longest_prompts_path = TABLES_DIR / "longest_prompts_by_word_count.csv"

distilbert_token_summary_path = TABLES_DIR / "distilbert_token_length_summary_by_split_label.csv"
roberta_token_summary_path = TABLES_DIR / "roberta_token_length_summary_by_split_label.csv"
secbert_token_summary_path = TABLES_DIR / "secbert_token_length_summary_by_split_label.csv"

overall_token_threshold_summary_path = TABLES_DIR / "overall_token_threshold_summary.csv"
token_threshold_summary_by_split_label_path = TABLES_DIR / "token_threshold_summary_by_split_label.csv"

text_length_analysis_df.to_csv(text_length_analysis_output_path, index=False)

character_length_summary_by_split_label_df.to_csv(character_length_summary_path, index=False)
word_count_summary_by_split_label_df.to_csv(word_count_summary_path, index=False)

shortest_prompts_df.to_csv(shortest_prompts_path, index=False)
longest_prompts_df.to_csv(longest_prompts_path, index=False)

distilbert_token_summary_df.to_csv(distilbert_token_summary_path, index=False)
roberta_token_summary_df.to_csv(roberta_token_summary_path, index=False)
secbert_token_summary_df.to_csv(secbert_token_summary_path, index=False)

overall_token_threshold_summary_df.to_csv(overall_token_threshold_summary_path, index=False)
token_threshold_summary_by_split_label_df.to_csv(
    token_threshold_summary_by_split_label_path,
    index=False
)

print("Text length analysis dataset saved to:", text_length_analysis_output_path)
print("File exists:", text_length_analysis_output_path.exists())

print("\nCharacter-length summary saved to:", character_length_summary_path)
print("File exists:", character_length_summary_path.exists())

print("\nWord-count summary saved to:", word_count_summary_path)
print("File exists:", word_count_summary_path.exists())

print("\nShortest prompts saved to:", shortest_prompts_path)
print("File exists:", shortest_prompts_path.exists())

print("\nLongest prompts saved to:", longest_prompts_path)
print("File exists:", longest_prompts_path.exists())

print("\nDistilBERT token summary saved to:", distilbert_token_summary_path)
print("File exists:", distilbert_token_summary_path.exists())

print("\nRoBERTa token summary saved to:", roberta_token_summary_path)
print("File exists:", roberta_token_summary_path.exists())

print("\nSecBERT token summary saved to:", secbert_token_summary_path)
print("File exists:", secbert_token_summary_path.exists())

print("\nOverall token threshold summary saved to:", overall_token_threshold_summary_path)
print("File exists:", overall_token_threshold_summary_path.exists())

print("\nToken threshold summary by split and label saved to:", token_threshold_summary_by_split_label_path)
print("File exists:", token_threshold_summary_by_split_label_path.exists())

#Part-10: Prompt Pattern Analysis

In [ ]:
# 1-Analyse common words in SAFE prompts

from collections import Counter
import re

prompt_pattern_analysis_df = combined_eda_df[["split", "label", "text"]].copy()
prompt_pattern_analysis_df["label_name"] = prompt_pattern_analysis_df["label"].map(label_mapping)

basic_stop_words = {
    "a", "an", "and", "are", "as", "at", "be", "by", "for", "from",
    "has", "have", "he", "her", "his", "i", "in", "is", "it", "its",
    "me", "my", "of", "on", "or", "our", "she", "that", "the", "their",
    "them", "they", "this", "to", "we", "what", "which", "who", "will",
    "with", "would", "you", "your", "can", "could", "should", "please",
    "about", "into", "than", "then", "there", "these", "those", "was",
    "were", "am", "do", "does", "did", "if", "so", "but", "not"
}

def extract_words_for_pattern_analysis(text):
    if not isinstance(text, str):
        return []

    text = text.lower()
    words = re.findall(r"\b[a-zA-Z][a-zA-Z]+\b", text)
    words = [word for word in words if word not in basic_stop_words]

    return words

safe_words = []

safe_texts = prompt_pattern_analysis_df[
    prompt_pattern_analysis_df["label"] == 0
]["text"]

for text in safe_texts:
    safe_words.extend(extract_words_for_pattern_analysis(text))

safe_common_words_df = pd.DataFrame(
    Counter(safe_words).most_common(30),
    columns=["word", "count"]
)

safe_common_words_df["label"] = 0
safe_common_words_df["label_name"] = "SAFE"

safe_common_words_df = safe_common_words_df[
    ["label", "label_name", "word", "count"]
]

print("Top common words in SAFE prompts:")
display(safe_common_words_df)

In [ ]:
# 2-Analyse common words in INJECTION prompts

injection_words = []

injection_texts = prompt_pattern_analysis_df[
    prompt_pattern_analysis_df["label"] == 1
]["text"]

for text in injection_texts:
    injection_words.extend(extract_words_for_pattern_analysis(text))

injection_common_words_df = pd.DataFrame(
    Counter(injection_words).most_common(30),
    columns=["word", "count"]
)

injection_common_words_df["label"] = 1
injection_common_words_df["label_name"] = "INJECTION"

injection_common_words_df = injection_common_words_df[
    ["label", "label_name", "word", "count"]
]

print("Top common words in INJECTION prompts:")
display(injection_common_words_df)

In [ ]:
# 3-Analyse common bigrams in SAFE prompts

def extract_bigrams_for_pattern_analysis(text):
    words = extract_words_for_pattern_analysis(text)
    bigrams = [" ".join(pair) for pair in zip(words, words[1:])]

    return bigrams

safe_bigrams = []

for text in safe_texts:
    safe_bigrams.extend(extract_bigrams_for_pattern_analysis(text))

safe_common_bigrams_df = pd.DataFrame(
    Counter(safe_bigrams).most_common(30),
    columns=["bigram", "count"]
)

safe_common_bigrams_df["label"] = 0
safe_common_bigrams_df["label_name"] = "SAFE"

safe_common_bigrams_df = safe_common_bigrams_df[
    ["label", "label_name", "bigram", "count"]
]

print("Top common bigrams in SAFE prompts:")
display(safe_common_bigrams_df)

In [ ]:
# 4-Analyse common bigrams in INJECTION prompts

injection_bigrams = []

for text in injection_texts:
    injection_bigrams.extend(extract_bigrams_for_pattern_analysis(text))

injection_common_bigrams_df = pd.DataFrame(
    Counter(injection_bigrams).most_common(30),
    columns=["bigram", "count"]
)

injection_common_bigrams_df["label"] = 1
injection_common_bigrams_df["label_name"] = "INJECTION"

injection_common_bigrams_df = injection_common_bigrams_df[
    ["label", "label_name", "bigram", "count"]
]

print("Top common bigrams in INJECTION prompts:")
display(injection_common_bigrams_df)

In [ ]:
# 5-Count prompt-injection terms

prompt_injection_term_patterns = {
    "ignore": r"\bignore\b",
    "previous": r"\bprevious\b",
    "instruction": r"\binstructions?\b",
    "system": r"\bsystem\b",
    "developer": r"\bdeveloper\b",
    "secret": r"\bsecret\b",
    "reveal": r"\breveal\b",
    "bypass": r"\bbypass\b",
    "override": r"\boverride\b",
    "jailbreak": r"\bjailbreak(?:ing)?\b",
    "forget": r"\bforget\b",
    "dan": r"\bdan\b",
    "chatgpt": r"\bchatgpt\b",
    "mode": r"\bmode\b"
}

def count_pattern_occurrences(text, pattern):
    if not isinstance(text, str):
        return 0

    return len(re.findall(pattern, text.lower()))

for term_name, term_pattern in prompt_injection_term_patterns.items():
    column_name = f"term_{term_name}_count"

    prompt_pattern_analysis_df[column_name] = prompt_pattern_analysis_df["text"].apply(
        lambda x: count_pattern_occurrences(x, term_pattern)
    )

term_count_rows = []

for label_value, label_group_df in prompt_pattern_analysis_df.groupby("label"):
    total_prompts = len(label_group_df)

    for term_name in prompt_injection_term_patterns.keys():
        column_name = f"term_{term_name}_count"

        total_occurrence_count = label_group_df[column_name].sum()
        prompt_count_with_term = (label_group_df[column_name] > 0).sum()
        prompt_percentage_with_term = (prompt_count_with_term / total_prompts) * 100

        term_count_rows.append(
            {
                "label": label_value,
                "label_name": label_mapping[label_value],
                "term": term_name,
                "total_prompts": total_prompts,
                "total_occurrence_count": total_occurrence_count,
                "prompt_count_with_term": prompt_count_with_term,
                "prompt_percentage_with_term": round(prompt_percentage_with_term, 2)
            }
        )

prompt_injection_term_counts_df = pd.DataFrame(term_count_rows)

print("Prompt-injection term counts by label:")
display(prompt_injection_term_counts_df)

In [ ]:
# 6-Count URLs

url_pattern = r"(https?://\S+|www\.\S+)"

prompt_pattern_analysis_df["url_count"] = prompt_pattern_analysis_df["text"].apply(
    lambda x: count_pattern_occurrences(x, url_pattern)
)

url_count_summary_df = (
    prompt_pattern_analysis_df
    .groupby("label")
    .agg(
        total_prompts=("text", "count"),
        total_url_count=("url_count", "sum"),
        prompt_count_with_url=("url_count", lambda x: (x > 0).sum())
    )
    .reset_index()
)

url_count_summary_df["label_name"] = url_count_summary_df["label"].map(label_mapping)

url_count_summary_df["prompt_percentage_with_url"] = (
    url_count_summary_df["prompt_count_with_url"] / url_count_summary_df["total_prompts"] * 100
).round(2)

url_count_summary_df = url_count_summary_df[
    [
        "label",
        "label_name",
        "total_prompts",
        "total_url_count",
        "prompt_count_with_url",
        "prompt_percentage_with_url"
    ]
]

url_examples_df = prompt_pattern_analysis_df[
    prompt_pattern_analysis_df["url_count"] > 0
].copy()

print("URL count summary by label:")
display(url_count_summary_df)

print("\nURL examples:")
display(
    url_examples_df[
        [
            "split",
            "label",
            "label_name",
            "text",
            "url_count"
        ]
    ].head(20)
)

In [ ]:
# 7-Count code-like patterns

code_like_pattern_definitions = {
    "programming_language_reference": r"\b(python|javascript|java|php|html|css|sql|bash|powershell|linux|terminal|code|program|script)\b|c\+\+|c#",
    "function_or_class_keyword": r"\b(def|function|class|return|import|print|console\.log)\b",
    "function_call_like_pattern": r"\b[a-zA-Z_][a-zA-Z0-9_]*\s*\(",
    "assignment_or_operator_pattern": r"(\w+\s*=\s*[^=]|==|!=|<=|>=|=>|->|\+\+|--)",
    "curly_brace_or_semicolon": r"[{};]",
    "html_or_xml_tag": r"</?[a-zA-Z][^>]*>",
    "code_file_extension": r"\b[\w\-]+\.(py|js|java|cpp|c|html|css|sql|sh|ps1|php|json|xml|yaml|yml)\b"
}

for pattern_name, pattern in code_like_pattern_definitions.items():
    column_name = f"code_pattern_{pattern_name}_count"

    prompt_pattern_analysis_df[column_name] = prompt_pattern_analysis_df["text"].apply(
        lambda x: count_pattern_occurrences(x, pattern)
    )

code_like_count_columns = [
    f"code_pattern_{pattern_name}_count"
    for pattern_name in code_like_pattern_definitions.keys()
]

prompt_pattern_analysis_df["code_like_total_count"] = prompt_pattern_analysis_df[
    code_like_count_columns
].sum(axis=1)

prompt_pattern_analysis_df["has_code_like_pattern"] = (
    prompt_pattern_analysis_df["code_like_total_count"] > 0
)

code_like_pattern_rows = []

for label_value, label_group_df in prompt_pattern_analysis_df.groupby("label"):
    total_prompts = len(label_group_df)

    for pattern_name in code_like_pattern_definitions.keys():
        column_name = f"code_pattern_{pattern_name}_count"

        total_occurrence_count = label_group_df[column_name].sum()
        prompt_count_with_pattern = (label_group_df[column_name] > 0).sum()
        prompt_percentage_with_pattern = (prompt_count_with_pattern / total_prompts) * 100

        code_like_pattern_rows.append(
            {
                "label": label_value,
                "label_name": label_mapping[label_value],
                "pattern_name": pattern_name,
                "total_prompts": total_prompts,
                "total_occurrence_count": total_occurrence_count,
                "prompt_count_with_pattern": prompt_count_with_pattern,
                "prompt_percentage_with_pattern": round(prompt_percentage_with_pattern, 2)
            }
        )

code_like_pattern_summary_df = pd.DataFrame(code_like_pattern_rows)

code_like_overall_summary_df = (
    prompt_pattern_analysis_df
    .groupby("label")
    .agg(
        total_prompts=("text", "count"),
        total_code_like_occurrence_count=("code_like_total_count", "sum"),
        prompt_count_with_any_code_like_pattern=("has_code_like_pattern", "sum")
    )
    .reset_index()
)

code_like_overall_summary_df["label_name"] = code_like_overall_summary_df["label"].map(label_mapping)

code_like_overall_summary_df["prompt_percentage_with_any_code_like_pattern"] = (
    code_like_overall_summary_df["prompt_count_with_any_code_like_pattern"]
    / code_like_overall_summary_df["total_prompts"]
    * 100
).round(2)

code_like_overall_summary_df = code_like_overall_summary_df[
    [
        "label",
        "label_name",
        "total_prompts",
        "total_code_like_occurrence_count",
        "prompt_count_with_any_code_like_pattern",
        "prompt_percentage_with_any_code_like_pattern"
    ]
]

code_like_examples_df = prompt_pattern_analysis_df[
    prompt_pattern_analysis_df["has_code_like_pattern"]
].copy()

code_like_examples_df = code_like_examples_df.sort_values(
    by="code_like_total_count",
    ascending=False
)

print("Overall code-like pattern summary by label:")
display(code_like_overall_summary_df)

print("\nDetailed code-like pattern summary by label:")
display(code_like_pattern_summary_df)

print("\nCode-like pattern examples:")
display(
    code_like_examples_df[
        [
            "split",
            "label",
            "label_name",
            "text",
            "code_like_total_count"
        ]
    ].head(20)
)

In [ ]:
# 8-Count command-like patterns

command_like_pattern_definitions = {
    "act_as_role_instruction": r"\b(act as|act like|pretend to be|roleplay as|you are now|you are going to act|from now on)\b",
    "ignore_or_forget_instruction": r"\b(ignore|forget|disregard|neglect)\b",
    "previous_instruction_reference": r"\b(previous instructions?|above instructions?|prior instructions?|all instructions?)\b",
    "new_task_or_override": r"\b(new task|your task is|instead|override|bypass)\b",
    "response_constraint": r"\b(only reply|do not reply|do not explain|without explanation|no explanation|just answer|answer only)\b",
    "execution_command": r"\b(execute|run|print|compile|generate|write|create|translate|summarise|summarize|answer|show|tell)\b",
    "system_or_prompt_reference": r"\b(system prompt|developer message|hidden prompt|secret prompt|prompt text|your prompt)\b",
    "output_format_command": r"\b(code block|json|markdown|terminal output|format|template)\b"
}

for pattern_name, pattern in command_like_pattern_definitions.items():
    column_name = f"command_pattern_{pattern_name}_count"

    prompt_pattern_analysis_df[column_name] = prompt_pattern_analysis_df["text"].apply(
        lambda x: count_pattern_occurrences(x, pattern)
    )

command_like_count_columns = [
    f"command_pattern_{pattern_name}_count"
    for pattern_name in command_like_pattern_definitions.keys()
]

prompt_pattern_analysis_df["command_like_total_count"] = prompt_pattern_analysis_df[
    command_like_count_columns
].sum(axis=1)

prompt_pattern_analysis_df["has_command_like_pattern"] = (
    prompt_pattern_analysis_df["command_like_total_count"] > 0
)

command_like_pattern_rows = []

for label_value, label_group_df in prompt_pattern_analysis_df.groupby("label"):
    total_prompts = len(label_group_df)

    for pattern_name in command_like_pattern_definitions.keys():
        column_name = f"command_pattern_{pattern_name}_count"

        total_occurrence_count = label_group_df[column_name].sum()
        prompt_count_with_pattern = (label_group_df[column_name] > 0).sum()
        prompt_percentage_with_pattern = (prompt_count_with_pattern / total_prompts) * 100

        command_like_pattern_rows.append(
            {
                "label": label_value,
                "label_name": label_mapping[label_value],
                "pattern_name": pattern_name,
                "total_prompts": total_prompts,
                "total_occurrence_count": total_occurrence_count,
                "prompt_count_with_pattern": prompt_count_with_pattern,
                "prompt_percentage_with_pattern": round(prompt_percentage_with_pattern, 2)
            }
        )

command_like_pattern_summary_df = pd.DataFrame(command_like_pattern_rows)

command_like_overall_summary_df = (
    prompt_pattern_analysis_df
    .groupby("label")
    .agg(
        total_prompts=("text", "count"),
        total_command_like_occurrence_count=("command_like_total_count", "sum"),
        prompt_count_with_any_command_like_pattern=("has_command_like_pattern", "sum")
    )
    .reset_index()
)

command_like_overall_summary_df["label_name"] = command_like_overall_summary_df["label"].map(label_mapping)

command_like_overall_summary_df["prompt_percentage_with_any_command_like_pattern"] = (
    command_like_overall_summary_df["prompt_count_with_any_command_like_pattern"]
    / command_like_overall_summary_df["total_prompts"]
    * 100
).round(2)

command_like_overall_summary_df = command_like_overall_summary_df[
    [
        "label",
        "label_name",
        "total_prompts",
        "total_command_like_occurrence_count",
        "prompt_count_with_any_command_like_pattern",
        "prompt_percentage_with_any_command_like_pattern"
    ]
]

command_like_examples_df = prompt_pattern_analysis_df[
    prompt_pattern_analysis_df["has_command_like_pattern"]
].copy()

command_like_examples_df = command_like_examples_df.sort_values(
    by="command_like_total_count",
    ascending=False
)

print("Overall command-like pattern summary by label:")
display(command_like_overall_summary_df)

print("\nDetailed command-like pattern summary by label:")
display(command_like_pattern_summary_df)

print("\nCommand-like pattern examples:")
display(
    command_like_examples_df[
        [
            "split",
            "label",
            "label_name",
            "text",
            "command_like_total_count"
        ]
    ].head(20)
)

In [ ]:
# 9-Compare pattern frequencies by label

pattern_frequency_comparison_rows = []

def add_pattern_comparison_rows(
    source_df,
    pattern_group,
    pattern_column,
    count_column,
    percentage_column
):
    for pattern_name in source_df[pattern_column].unique():
        pattern_df = source_df[source_df[pattern_column] == pattern_name]

        safe_row = pattern_df[pattern_df["label"] == 0]
        injection_row = pattern_df[pattern_df["label"] == 1]

        safe_count = int(safe_row[count_column].iloc[0]) if len(safe_row) > 0 else 0
        injection_count = int(injection_row[count_column].iloc[0]) if len(injection_row) > 0 else 0

        safe_percentage = float(safe_row[percentage_column].iloc[0]) if len(safe_row) > 0 else 0.0
        injection_percentage = float(injection_row[percentage_column].iloc[0]) if len(injection_row) > 0 else 0.0

        pattern_frequency_comparison_rows.append(
            {
                "pattern_group": pattern_group,
                "pattern_name": pattern_name,
                "safe_prompt_count": safe_count,
                "injection_prompt_count": injection_count,
                "safe_percentage": safe_percentage,
                "injection_percentage": injection_percentage,
                "injection_minus_safe_percentage": round(
                    injection_percentage - safe_percentage,
                    2
                )
            }
        )

# Compare prompt-injection terms
add_pattern_comparison_rows(
    source_df=prompt_injection_term_counts_df,
    pattern_group="prompt_injection_term",
    pattern_column="term",
    count_column="prompt_count_with_term",
    percentage_column="prompt_percentage_with_term"
)

# Compare URL presence
url_pattern_comparison_source_df = url_count_summary_df.copy()
url_pattern_comparison_source_df["pattern_name"] = "url"

add_pattern_comparison_rows(
    source_df=url_pattern_comparison_source_df,
    pattern_group="url",
    pattern_column="pattern_name",
    count_column="prompt_count_with_url",
    percentage_column="prompt_percentage_with_url"
)

# Compare any code-like pattern
code_like_any_comparison_source_df = code_like_overall_summary_df.copy()
code_like_any_comparison_source_df["pattern_name"] = "any_code_like_pattern"

add_pattern_comparison_rows(
    source_df=code_like_any_comparison_source_df,
    pattern_group="code_like_overall",
    pattern_column="pattern_name",
    count_column="prompt_count_with_any_code_like_pattern",
    percentage_column="prompt_percentage_with_any_code_like_pattern"
)

# Compare detailed code-like patterns
add_pattern_comparison_rows(
    source_df=code_like_pattern_summary_df,
    pattern_group="code_like_detail",
    pattern_column="pattern_name",
    count_column="prompt_count_with_pattern",
    percentage_column="prompt_percentage_with_pattern"
)

# Compare any command-like pattern
command_like_any_comparison_source_df = command_like_overall_summary_df.copy()
command_like_any_comparison_source_df["pattern_name"] = "any_command_like_pattern"

add_pattern_comparison_rows(
    source_df=command_like_any_comparison_source_df,
    pattern_group="command_like_overall",
    pattern_column="pattern_name",
    count_column="prompt_count_with_any_command_like_pattern",
    percentage_column="prompt_percentage_with_any_command_like_pattern"
)

# Compare detailed command-like patterns
add_pattern_comparison_rows(
    source_df=command_like_pattern_summary_df,
    pattern_group="command_like_detail",
    pattern_column="pattern_name",
    count_column="prompt_count_with_pattern",
    percentage_column="prompt_percentage_with_pattern"
)

pattern_frequency_comparison_df = pd.DataFrame(pattern_frequency_comparison_rows)

pattern_frequency_comparison_df = pattern_frequency_comparison_df.sort_values(
    by="injection_minus_safe_percentage",
    ascending=False
).reset_index(drop=True)

print("Pattern frequency comparison by label:")
display(pattern_frequency_comparison_df)

print("\nTop patterns more frequent in INJECTION prompts:")
display(pattern_frequency_comparison_df.head(20))

In [ ]:
# 10-Save word, phrase, and suspicious-pattern summaries

prompt_pattern_analysis_path = TABLES_DIR / "prompt_pattern_analysis_with_flags.csv"

safe_common_words_path = TABLES_DIR / "safe_common_words.csv"
injection_common_words_path = TABLES_DIR / "injection_common_words.csv"

safe_common_bigrams_path = TABLES_DIR / "safe_common_bigrams.csv"
injection_common_bigrams_path = TABLES_DIR / "injection_common_bigrams.csv"

prompt_injection_term_counts_path = TABLES_DIR / "prompt_injection_term_counts_by_label.csv"

url_count_summary_path = TABLES_DIR / "url_count_summary_by_label.csv"
url_examples_path = TABLES_DIR / "url_examples.csv"

code_like_overall_summary_path = TABLES_DIR / "code_like_overall_summary_by_label.csv"
code_like_pattern_summary_path = TABLES_DIR / "code_like_pattern_summary_by_label.csv"
code_like_examples_path = TABLES_DIR / "code_like_examples.csv"

command_like_overall_summary_path = TABLES_DIR / "command_like_overall_summary_by_label.csv"
command_like_pattern_summary_path = TABLES_DIR / "command_like_pattern_summary_by_label.csv"
command_like_examples_path = TABLES_DIR / "command_like_examples.csv"

pattern_frequency_comparison_path = TABLES_DIR / "pattern_frequency_comparison_by_label.csv"

prompt_pattern_analysis_df.to_csv(prompt_pattern_analysis_path, index=False)

safe_common_words_df.to_csv(safe_common_words_path, index=False)
injection_common_words_df.to_csv(injection_common_words_path, index=False)

safe_common_bigrams_df.to_csv(safe_common_bigrams_path, index=False)
injection_common_bigrams_df.to_csv(injection_common_bigrams_path, index=False)

prompt_injection_term_counts_df.to_csv(prompt_injection_term_counts_path, index=False)

url_count_summary_df.to_csv(url_count_summary_path, index=False)
url_examples_df.to_csv(url_examples_path, index=False)

code_like_overall_summary_df.to_csv(code_like_overall_summary_path, index=False)
code_like_pattern_summary_df.to_csv(code_like_pattern_summary_path, index=False)
code_like_examples_df.to_csv(code_like_examples_path, index=False)

command_like_overall_summary_df.to_csv(command_like_overall_summary_path, index=False)
command_like_pattern_summary_df.to_csv(command_like_pattern_summary_path, index=False)
command_like_examples_df.to_csv(command_like_examples_path, index=False)

pattern_frequency_comparison_df.to_csv(pattern_frequency_comparison_path, index=False)

saved_pattern_files = [
    prompt_pattern_analysis_path,
    safe_common_words_path,
    injection_common_words_path,
    safe_common_bigrams_path,
    injection_common_bigrams_path,
    prompt_injection_term_counts_path,
    url_count_summary_path,
    url_examples_path,
    code_like_overall_summary_path,
    code_like_pattern_summary_path,
    code_like_examples_path,
    command_like_overall_summary_path,
    command_like_pattern_summary_path,
    command_like_examples_path,
    pattern_frequency_comparison_path
]

print("Phase 10 pattern-analysis files saved:")

for file_path in saved_pattern_files:
    print(file_path)
    print("File exists:", file_path.exists())
    print("-" * 70)

#Part-11: Outlier and Data Quality Risk Analysis

In [ ]:
# 1-Review shortest prompts

if "shortest_prompts_df" not in globals():
    shortest_prompts_df = pd.read_csv(TABLES_DIR / "shortest_prompts_by_word_count.csv")

shortest_prompt_review_df = shortest_prompts_df.copy()

if "label_name" not in shortest_prompt_review_df.columns:
    shortest_prompt_review_df["label_name"] = shortest_prompt_review_df["label"].map(label_mapping)

shortest_prompt_label_summary_df = (
    shortest_prompt_review_df
    .groupby(["label", "label_name"])
    .agg(
        reviewed_prompt_count=("text", "count"),
        min_character_length=("character_length", "min"),
        max_character_length=("character_length", "max"),
        min_word_count=("word_count", "min"),
        max_word_count=("word_count", "max")
    )
    .reset_index()
)

print("Shortest prompt review summary:")
display(shortest_prompt_label_summary_df)

print("\nShortest prompt examples for manual review:")
display(
    shortest_prompt_review_df[
        [
            "split",
            "label",
            "label_name",
            "text",
            "character_length",
            "word_count"
        ]
    ]
)

print("\nReview note:")
print("These prompts are reviewed for data-quality risk only.")
print("No labels are changed in this step.")

In [ ]:
# 2-Review longest prompts

if "longest_prompts_df" not in globals():
    longest_prompts_df = pd.read_csv(TABLES_DIR / "longest_prompts_by_word_count.csv")

longest_prompt_review_df = longest_prompts_df.copy()

if "label_name" not in longest_prompt_review_df.columns:
    longest_prompt_review_df["label_name"] = longest_prompt_review_df["label"].map(label_mapping)

longest_prompt_label_summary_df = (
    longest_prompt_review_df
    .groupby(["label", "label_name"])
    .agg(
        reviewed_prompt_count=("text", "count"),
        min_character_length=("character_length", "min"),
        max_character_length=("character_length", "max"),
        min_word_count=("word_count", "min"),
        max_word_count=("word_count", "max")
    )
    .reset_index()
)

print("Longest prompt review summary:")
display(longest_prompt_label_summary_df)

print("\nLongest prompt examples for manual review:")
display(
    longest_prompt_review_df[
        [
            "split",
            "label",
            "label_name",
            "text",
            "character_length",
            "word_count"
        ]
    ]
)

print("\nReview note:")
print("These prompts are reviewed for long-text and truncation risk.")
print("No labels are changed in this step.")

In [ ]:
# 3-Review extreme token-length prompts

if "text_length_analysis_df" not in globals():
    text_length_analysis_df = pd.read_csv(
        TABLES_DIR / "text_length_analysis_with_token_lengths.csv"
    )

token_length_columns = [
    "distilbert_token_length",
    "roberta_token_length",
    "secbert_token_length"
]

text_length_analysis_df["max_token_length_across_models"] = text_length_analysis_df[
    token_length_columns
].max(axis=1)

text_length_analysis_df["exceeds_512_tokens_any_model"] = (
    text_length_analysis_df["max_token_length_across_models"] > 512
)

extreme_token_length_review_df = text_length_analysis_df[
    text_length_analysis_df["exceeds_512_tokens_any_model"]
].copy()

extreme_token_length_review_df["label_name"] = extreme_token_length_review_df["label"].map(label_mapping)

extreme_token_length_review_df = extreme_token_length_review_df.sort_values(
    by="max_token_length_across_models",
    ascending=False
)

extreme_token_length_summary_df = (
    extreme_token_length_review_df
    .groupby(["label", "label_name"])
    .agg(
        extreme_prompt_count=("text", "count"),
        max_distilbert_tokens=("distilbert_token_length", "max"),
        max_roberta_tokens=("roberta_token_length", "max"),
        max_secbert_tokens=("secbert_token_length", "max"),
        max_token_length_across_models=("max_token_length_across_models", "max")
    )
    .reset_index()
)

print("Extreme token-length prompt summary:")
display(extreme_token_length_summary_df)

print("\nExtreme token-length prompt examples:")
display(
    extreme_token_length_review_df[
        [
            "split",
            "label",
            "label_name",
            "text",
            "character_length",
            "word_count",
            "distilbert_token_length",
            "roberta_token_length",
            "secbert_token_length",
            "max_token_length_across_models"
        ]
    ]
)

print("\nReview note:")
print("These prompts exceed 512 tokens for at least one planned transformer tokenizer.")
print("This indicates truncation risk during model fine-tuning.")
print("No rows or labels are changed in this step.")

In [ ]:
# 4-Review unusual-character examples

if "unusual_character_examples_df" not in globals():
    unusual_character_examples_df = pd.read_csv(
        TABLES_DIR / "unusual_character_examples.csv"
    )

unusual_character_review_df = unusual_character_examples_df.copy()

if "label_name" not in unusual_character_review_df.columns:
    unusual_character_review_df["label_name"] = unusual_character_review_df["label"].map(label_mapping)

unusual_character_review_summary_df = (
    unusual_character_review_df
    .groupby(["split", "label", "label_name"])
    .agg(
        reviewed_prompt_count=("text", "count")
    )
    .reset_index()
)

print("Unusual-character review summary:")
display(unusual_character_review_summary_df)

print("\nUnusual-character examples for manual review:")
display(unusual_character_review_df.head(20))

print("\nReview note:")
print("These prompts are reviewed for hidden/control/unusual-character risk.")
print("No rows or labels are changed in this step.")

In [ ]:
# 5-Review duplicate and conflicting examples

duplicate_conflicting_file_paths = {
    "duplicate_full_rows": TABLES_DIR / "duplicate_full_row_examples.csv",
    "duplicate_text_examples": TABLES_DIR / "duplicate_text_examples.csv",
    "conflicting_label_examples": TABLES_DIR / "conflicting_label_examples.csv",
    "exact_train_test_overlap": TABLES_DIR / "exact_train_test_overlap_examples.csv",
    "normalised_duplicate_text_examples": TABLES_DIR / "normalised_duplicate_text_examples.csv",
    "normalised_train_test_overlap": TABLES_DIR / "normalised_train_test_overlap_examples.csv",
    "normalised_conflicting_label_examples": TABLES_DIR / "normalised_conflicting_label_examples.csv"
}

duplicate_conflicting_review_tables = {}
duplicate_conflicting_review_rows = []

for review_name, file_path in duplicate_conflicting_file_paths.items():
    if file_path.exists():
        review_df = pd.read_csv(file_path)
    else:
        review_df = pd.DataFrame()

    duplicate_conflicting_review_tables[review_name] = review_df

    duplicate_conflicting_review_rows.append(
        {
            "review_item": review_name,
            "file_path": str(file_path),
            "file_exists": file_path.exists(),
            "row_count": len(review_df)
        }
    )

duplicate_conflicting_review_summary_df = pd.DataFrame(duplicate_conflicting_review_rows)

print("Duplicate/conflicting review summary:")
display(duplicate_conflicting_review_summary_df)

print("\nDuplicate/conflicting examples for manual review:")

for review_name, review_df in duplicate_conflicting_review_tables.items():
    print("\n" + "=" * 80)
    print(review_name)
    print("Rows:", len(review_df))

    if len(review_df) > 0:
        display(review_df.head(20))
    else:
        print("No examples found.")

print("\nReview note:")
print("These files are reviewed for duplicate, conflicting-label, and train/test leakage risk.")
print("No rows or labels are changed in this step.")

In [ ]:
# 6-Review possible label-noise examples without changing labels

possible_label_noise_source_df = combined_eda_df[["split", "label", "text"]].copy()
possible_label_noise_source_df = possible_label_noise_source_df.reset_index().rename(
    columns={"index": "combined_row_index"}
)

possible_label_noise_source_df["label_name"] = possible_label_noise_source_df["label"].map(label_mapping)

# Add word count if available
if "text_length_analysis_df" in globals():
    text_length_for_noise_df = text_length_analysis_df[["split", "label", "text", "character_length", "word_count"]].copy()
else:
    text_length_for_noise_df = pd.read_csv(
        TABLES_DIR / "text_length_analysis_with_token_lengths.csv"
    )[["split", "label", "text", "character_length", "word_count"]].copy()

possible_label_noise_source_df = possible_label_noise_source_df.merge(
    text_length_for_noise_df,
    on=["split", "label", "text"],
    how="left"
)

# Add pattern flags if available
if "prompt_pattern_analysis_df" in globals():
    pattern_for_noise_df = prompt_pattern_analysis_df.copy()
else:
    pattern_for_noise_df = pd.read_csv(
        TABLES_DIR / "prompt_pattern_analysis_with_flags.csv"
    )

pattern_columns_for_noise = [
    "url_count",
    "code_like_total_count",
    "has_code_like_pattern",
    "command_like_total_count",
    "has_command_like_pattern"
]

term_columns_for_noise = [
    column_name for column_name in pattern_for_noise_df.columns
    if column_name.startswith("term_") and column_name.endswith("_count")
]

pattern_for_noise_df["prompt_injection_term_total_count"] = pattern_for_noise_df[
    term_columns_for_noise
].sum(axis=1)

possible_label_noise_source_df = possible_label_noise_source_df.merge(
    pattern_for_noise_df[
        ["split", "label", "text", "prompt_injection_term_total_count"] + pattern_columns_for_noise
    ],
    on=["split", "label", "text"],
    how="left"
)

# Add unusual-character flag if available
if "unusual_character_examples_df" in globals():
    unusual_for_noise_df = unusual_character_examples_df[["split", "label", "text", "has_any_unusual_character"]].copy()
else:
    unusual_for_noise_df = pd.read_csv(
        TABLES_DIR / "unusual_character_examples.csv"
    )[["split", "label", "text", "has_any_unusual_character"]].copy()

possible_label_noise_source_df = possible_label_noise_source_df.merge(
    unusual_for_noise_df,
    on=["split", "label", "text"],
    how="left"
)

possible_label_noise_source_df["has_any_unusual_character"] = (
    possible_label_noise_source_df["has_any_unusual_character"].fillna(False)
)

candidate_frames = []

def add_label_noise_candidates(condition, reason):
    candidate_df = possible_label_noise_source_df[condition].copy()
    candidate_df["review_reason"] = reason
    candidate_frames.append(candidate_df)

# INJECTION prompts that are extremely short
add_label_noise_candidates(
    (possible_label_noise_source_df["label"] == 1)
    & (possible_label_noise_source_df["word_count"] <= 3),
    "INJECTION label with extremely short text"
)

# SAFE prompts containing injection-specific terms
add_label_noise_candidates(
    (possible_label_noise_source_df["label"] == 0)
    & (possible_label_noise_source_df["prompt_injection_term_total_count"] > 0),
    "SAFE label contains prompt-injection term"
)

# SAFE prompts containing command-like patterns
add_label_noise_candidates(
    (possible_label_noise_source_df["label"] == 0)
    & (possible_label_noise_source_df["has_command_like_pattern"] == True),
    "SAFE label contains command-like pattern"
)

# SAFE prompts containing code-like patterns
add_label_noise_candidates(
    (possible_label_noise_source_df["label"] == 0)
    & (possible_label_noise_source_df["has_code_like_pattern"] == True),
    "SAFE label contains code-like pattern"
)

# SAFE prompts containing unusual characters
add_label_noise_candidates(
    (possible_label_noise_source_df["label"] == 0)
    & (possible_label_noise_source_df["has_any_unusual_character"] == True),
    "SAFE label contains unusual or invisible character"
)

possible_label_noise_examples_df = pd.concat(candidate_frames, ignore_index=True)

possible_label_noise_examples_df = possible_label_noise_examples_df.sort_values(
    by=["label", "word_count", "character_length"],
    ascending=[True, True, True]
).reset_index(drop=True)

possible_label_noise_summary_df = (
    possible_label_noise_examples_df
    .groupby(["label", "label_name", "review_reason"])
    .agg(
        candidate_count=("text", "count")
    )
    .reset_index()
)

print("Possible label-noise candidate summary:")
display(possible_label_noise_summary_df)

print("\nPossible label-noise examples for manual review:")
display(
    possible_label_noise_examples_df[
        [
            "combined_row_index",
            "split",
            "label",
            "label_name",
            "review_reason",
            "text",
            "character_length",
            "word_count",
            "prompt_injection_term_total_count",
            "code_like_total_count",
            "command_like_total_count",
            "has_any_unusual_character"
        ]
    ].head(30)
)

print("\nReview note:")
print("These are only possible review candidates.")
print("No labels are changed in this step.")

In [ ]:
# 7-Summarise missing-value risk

if "data_quality_risk_rows" not in globals():
    data_quality_risk_rows = []

missing_value_summary_rows = []

datasets_for_missing_review = {
    "train": train_df,
    "test": test_df,
    "combined_eda": combined_eda_df
}

for dataset_name, dataset_df in datasets_for_missing_review.items():
    for column_name in ["text", "label"]:
        missing_count = dataset_df[column_name].isna().sum()
        total_rows = len(dataset_df)
        missing_percentage = (missing_count / total_rows) * 100

        missing_value_summary_rows.append(
            {
                "dataset": dataset_name,
                "column": column_name,
                "total_rows": total_rows,
                "missing_count": int(missing_count),
                "missing_percentage": round(missing_percentage, 2)
            }
        )

missing_value_summary_df = pd.DataFrame(missing_value_summary_rows)

total_missing_values = int(missing_value_summary_df["missing_count"].sum())

missing_value_risk_level = "Low" if total_missing_values == 0 else "High"

missing_value_risk_note = (
    "No missing values were found in text or label columns."
    if total_missing_values == 0
    else "Missing values were found and must be handled before model training."
)

data_quality_risk_rows = [
    row for row in data_quality_risk_rows
    if row["risk_area"] != "missing_values"
]

data_quality_risk_rows.append(
    {
        "risk_area": "missing_values",
        "risk_level": missing_value_risk_level,
        "evidence": f"Total missing values across train, test, and combined EDA checks: {total_missing_values}",
        "part_2_action": (
            "Keep validation check in data preparation. No imputation or row removal required based on current data."
            if total_missing_values == 0
            else "Handle missing text/label values before tokenisation and training."
        ),
        "notes": missing_value_risk_note
    }
)

print("Missing-value summary:")
display(missing_value_summary_df)

print("\nMissing-value risk summary:")
display(pd.DataFrame([data_quality_risk_rows[-1]]))

In [ ]:
# 8-Summarise empty and invalid text risk

empty_invalid_text_summary_rows = []

datasets_for_empty_invalid_review = {
    "train": train_df,
    "test": test_df,
    "combined_eda": combined_eda_df
}

def is_empty_text(value):
    return isinstance(value, str) and value == ""

def is_whitespace_only_text(value):
    return isinstance(value, str) and value.strip() == "" and value != ""

def is_non_string_text(value):
    return not isinstance(value, str)

def is_punctuation_or_symbol_only_text(value):
    if not isinstance(value, str):
        return False

    stripped_value = value.strip()

    if stripped_value == "":
        return False

    return re.fullmatch(r"[^A-Za-z0-9]+", stripped_value) is not None

for dataset_name, dataset_df in datasets_for_empty_invalid_review.items():
    total_rows = len(dataset_df)

    empty_text_count = dataset_df["text"].apply(is_empty_text).sum()
    whitespace_only_text_count = dataset_df["text"].apply(is_whitespace_only_text).sum()
    non_string_text_count = dataset_df["text"].apply(is_non_string_text).sum()
    punctuation_or_symbol_only_text_count = dataset_df["text"].apply(
        is_punctuation_or_symbol_only_text
    ).sum()

    empty_invalid_text_summary_rows.append(
        {
            "dataset": dataset_name,
            "total_rows": total_rows,
            "empty_text_count": int(empty_text_count),
            "whitespace_only_text_count": int(whitespace_only_text_count),
            "non_string_text_count": int(non_string_text_count),
            "punctuation_or_symbol_only_text_count": int(punctuation_or_symbol_only_text_count)
        }
    )

empty_invalid_text_summary_df = pd.DataFrame(empty_invalid_text_summary_rows)

total_empty_invalid_text_count = int(
    empty_invalid_text_summary_df[
        [
            "empty_text_count",
            "whitespace_only_text_count",
            "non_string_text_count",
            "punctuation_or_symbol_only_text_count"
        ]
    ].sum().sum()
)

empty_invalid_text_risk_level = "Low" if total_empty_invalid_text_count == 0 else "Medium"

empty_invalid_text_risk_note = (
    "No empty, whitespace-only, non-string, or punctuation-only text values were found."
    if total_empty_invalid_text_count == 0
    else "Some empty or invalid text values were found and should be reviewed before model training."
)

data_quality_risk_rows = [
    row for row in data_quality_risk_rows
    if row["risk_area"] != "empty_invalid_text"
]

data_quality_risk_rows.append(
    {
        "risk_area": "empty_invalid_text",
        "risk_level": empty_invalid_text_risk_level,
        "evidence": f"Total empty/invalid text indicators across train, test, and combined EDA checks: {total_empty_invalid_text_count}",
        "part_2_action": (
            "Keep validation check in data preparation. No row removal required based on current data."
            if total_empty_invalid_text_count == 0
            else "Review and handle invalid text values before tokenisation and training."
        ),
        "notes": empty_invalid_text_risk_note
    }
)

print("Empty/invalid-text summary:")
display(empty_invalid_text_summary_df)

print("\nEmpty/invalid-text risk summary:")
display(pd.DataFrame([data_quality_risk_rows[-1]]))

In [ ]:
# 9-Summarise duplicate risk

if "duplicate_leakage_summary_df" not in globals():
    duplicate_leakage_summary_df = pd.read_csv(
        TABLES_DIR / "duplicate_leakage_summary.csv"
    )

duplicate_risk_checks = [
    "train_duplicate_full_row_count",
    "test_duplicate_full_row_count",
    "combined_duplicate_full_row_count",
    "train_duplicate_text_count",
    "test_duplicate_text_count",
    "combined_duplicate_text_count",
    "train_duplicate_text_same_label_count",
    "test_duplicate_text_same_label_count",
    "combined_duplicate_text_same_label_count",
    "normalised_duplicate_text_count"
]

duplicate_risk_summary_df = duplicate_leakage_summary_df[
    duplicate_leakage_summary_df["check"].isin(duplicate_risk_checks)
].copy()

total_duplicate_risk_count = int(duplicate_risk_summary_df["count"].sum())

duplicate_risk_level = "Low" if total_duplicate_risk_count == 0 else "Medium"

duplicate_risk_note = (
    "No exact or normalised duplicate rows/texts were found."
    if total_duplicate_risk_count == 0
    else "Duplicate examples were found and should be handled before model training."
)

data_quality_risk_rows = [
    row for row in data_quality_risk_rows
    if row["risk_area"] != "duplicates"
]

data_quality_risk_rows.append(
    {
        "risk_area": "duplicates",
        "risk_level": duplicate_risk_level,
        "evidence": f"Total duplicate-related counts from duplicate/leakage checks: {total_duplicate_risk_count}",
        "part_2_action": (
            "Keep duplicate validation check in data preparation. No deduplication required based on current data."
            if total_duplicate_risk_count == 0
            else "Review duplicate examples and deduplicate before model training."
        ),
        "notes": duplicate_risk_note
    }
)

print("Duplicate-risk summary:")
display(duplicate_risk_summary_df)

print("\nDuplicate-risk final summary:")
display(pd.DataFrame([data_quality_risk_rows[-1]]))

In [ ]:
# 10-Summarise leakage risk

if "duplicate_leakage_summary_df" not in globals():
    duplicate_leakage_summary_df = pd.read_csv(
        TABLES_DIR / "duplicate_leakage_summary.csv"
    )

leakage_risk_checks = [
    "exact_train_test_overlapping_text_count",
    "exact_train_test_overlap_row_count",
    "normalised_train_test_overlapping_text_count",
    "normalised_train_test_overlap_row_count"
]

leakage_risk_summary_df = duplicate_leakage_summary_df[
    duplicate_leakage_summary_df["check"].isin(leakage_risk_checks)
].copy()

total_leakage_risk_count = int(leakage_risk_summary_df["count"].sum())

leakage_risk_level = "Low" if total_leakage_risk_count == 0 else "High"

leakage_risk_note = (
    "No exact or normalised train/test text overlap was found."
    if total_leakage_risk_count == 0
    else "Train/test overlap was found and must be handled before model training."
)

data_quality_risk_rows = [
    row for row in data_quality_risk_rows
    if row["risk_area"] != "train_test_leakage"
]

data_quality_risk_rows.append(
    {
        "risk_area": "train_test_leakage",
        "risk_level": leakage_risk_level,
        "evidence": f"Total leakage-related counts from duplicate/leakage checks: {total_leakage_risk_count}",
        "part_2_action": (
            "Keep train/test leakage validation check in data preparation. No leakage correction required based on current data."
            if total_leakage_risk_count == 0
            else "Remove or repartition overlapping examples before model training."
        ),
        "notes": leakage_risk_note
    }
)

print("Leakage-risk summary:")
display(leakage_risk_summary_df)

print("\nLeakage-risk final summary:")
display(pd.DataFrame([data_quality_risk_rows[-1]]))

In [ ]:
# 11-Summarise class-imbalance risk

class_imbalance_summary_rows = []

datasets_for_class_imbalance_review = {
    "train": train_df,
    "test": test_df,
    "combined_eda": combined_eda_df
}

for dataset_name, dataset_df in datasets_for_class_imbalance_review.items():
    total_rows = len(dataset_df)
    label_counts = dataset_df["label"].value_counts().reindex([0, 1], fill_value=0)

    for label_value, label_count in label_counts.items():
        label_percentage = (label_count / total_rows) * 100

        class_imbalance_summary_rows.append(
            {
                "dataset": dataset_name,
                "label": label_value,
                "label_name": label_mapping[label_value],
                "total_rows": total_rows,
                "label_count": int(label_count),
                "label_percentage": round(label_percentage, 2)
            }
        )

class_imbalance_summary_df = pd.DataFrame(class_imbalance_summary_rows)

train_label_percentages = class_imbalance_summary_df[
    class_imbalance_summary_df["dataset"] == "train"
]["label_percentage"].tolist()

train_minority_percentage = min(train_label_percentages)
train_majority_percentage = max(train_label_percentages)

if train_minority_percentage < 20:
    class_imbalance_risk_level = "High"
elif train_minority_percentage < 40:
    class_imbalance_risk_level = "Medium"
else:
    class_imbalance_risk_level = "Low"

class_imbalance_risk_note = (
    f"Training split is moderately imbalanced. Minority class percentage: {train_minority_percentage}%. "
    f"Majority class percentage: {train_majority_percentage}%."
)

data_quality_risk_rows = [
    row for row in data_quality_risk_rows
    if row["risk_area"] != "class_imbalance"
]

data_quality_risk_rows.append(
    {
        "risk_area": "class_imbalance",
        "risk_level": class_imbalance_risk_level,
        "evidence": (
            f"Train minority class percentage: {train_minority_percentage}%; "
            f"train majority class percentage: {train_majority_percentage}%."
        ),
        "part_2_action": (
            "Use stratified validation splitting. Monitor Precision, Recall, F1-score, and AUC-ROC. "
            "Consider class weighting only if model performance shows bias toward the majority class."
        ),
        "notes": class_imbalance_risk_note
    }
)

print("Class-imbalance summary:")
display(class_imbalance_summary_df)

print("\nClass-imbalance final summary:")
display(pd.DataFrame([data_quality_risk_rows[-1]]))

In [ ]:
# 12-Summarise long-text and truncation risk

if "overall_token_threshold_summary_df" not in globals():
    overall_token_threshold_summary_df = pd.read_csv(
        TABLES_DIR / "overall_token_threshold_summary.csv"
    )

if "extreme_token_length_review_df" not in globals():
    text_length_analysis_df = pd.read_csv(
        TABLES_DIR / "text_length_analysis_with_token_lengths.csv"
    )

    token_length_columns = [
        "distilbert_token_length",
        "roberta_token_length",
        "secbert_token_length"
    ]

    text_length_analysis_df["max_token_length_across_models"] = text_length_analysis_df[
        token_length_columns
    ].max(axis=1)

    text_length_analysis_df["exceeds_512_tokens_any_model"] = (
        text_length_analysis_df["max_token_length_across_models"] > 512
    )

    extreme_token_length_review_df = text_length_analysis_df[
        text_length_analysis_df["exceeds_512_tokens_any_model"]
    ].copy()

long_text_threshold_512_df = overall_token_threshold_summary_df[
    overall_token_threshold_summary_df["threshold"] == 512
].copy()

max_longer_than_512_count = int(
    long_text_threshold_512_df["longer_than_threshold_count"].max()
)

max_longer_than_512_percentage = float(
    long_text_threshold_512_df["longer_than_threshold_percentage"].max()
)

extreme_prompt_count_any_model = len(extreme_token_length_review_df)

if extreme_prompt_count_any_model == 0:
    long_text_truncation_risk_level = "Low"
elif max_longer_than_512_percentage < 5:
    long_text_truncation_risk_level = "Medium"
else:
    long_text_truncation_risk_level = "High"

long_text_truncation_risk_note = (
    f"{extreme_prompt_count_any_model} prompts exceed 512 tokens for at least one tokenizer. "
    f"The highest model-specific percentage above 512 tokens is {max_longer_than_512_percentage}%."
)

data_quality_risk_rows = [
    row for row in data_quality_risk_rows
    if row["risk_area"] != "long_text_truncation"
]

data_quality_risk_rows.append(
    {
        "risk_area": "long_text_truncation",
        "risk_level": long_text_truncation_risk_level,
        "evidence": (
            f"Prompts exceeding 512 tokens for at least one model: {extreme_prompt_count_any_model}; "
            f"maximum model-specific count above 512 tokens: {max_longer_than_512_count}; "
            f"maximum model-specific percentage above 512 tokens: {max_longer_than_512_percentage}%."
        ),
        "part_2_action": (
            "Use max_length=512 with truncation=True during tokenisation. "
            "Record that a very small number of long prompts may be truncated."
        ),
        "notes": long_text_truncation_risk_note
    }
)

print("Long-text/truncation threshold summary for 512 tokens:")
display(long_text_threshold_512_df)

print("\nExtreme token-length examples:")
display(
    extreme_token_length_review_df[
        [
            "split",
            "label",
            "text",
            "character_length",
            "word_count",
            "distilbert_token_length",
            "roberta_token_length",
            "secbert_token_length",
            "max_token_length_across_models"
        ]
    ]
)

print("\nLong-text/truncation final summary:")
display(pd.DataFrame([data_quality_risk_rows[-1]]))

In [ ]:
# 13-Summarise unusual-character risk

if "unusual_character_examples_df" not in globals():
    unusual_character_examples_df = pd.read_csv(
        TABLES_DIR / "unusual_character_examples.csv"
    )

unusual_character_risk_df = unusual_character_examples_df.copy()

if "label_name" not in unusual_character_risk_df.columns:
    unusual_character_risk_df["label_name"] = unusual_character_risk_df["label"].map(label_mapping)

total_dataset_rows = len(combined_eda_df)
total_unusual_character_examples = len(unusual_character_risk_df)

unusual_character_percentage = (
    total_unusual_character_examples / total_dataset_rows * 100
)

unusual_character_risk_summary_by_label_df = (
    unusual_character_risk_df
    .groupby(["label", "label_name"])
    .agg(
        unusual_character_prompt_count=("text", "count")
    )
    .reset_index()
)

unusual_character_risk_summary_by_split_df = (
    unusual_character_risk_df
    .groupby("split")
    .agg(
        unusual_character_prompt_count=("text", "count")
    )
    .reset_index()
)

unusual_character_type_columns = [
    "has_newline",
    "has_tab",
    "has_carriage_return",
    "has_repeated_space",
    "has_other_control_character",
    "has_zero_width_character"
]

unusual_character_type_summary_rows = []

for column_name in unusual_character_type_columns:
    if column_name in unusual_character_risk_df.columns:
        unusual_character_type_summary_rows.append(
            {
                "unusual_character_type": column_name,
                "prompt_count": int(unusual_character_risk_df[column_name].sum())
            }
        )

unusual_character_type_summary_df = pd.DataFrame(unusual_character_type_summary_rows)

if total_unusual_character_examples == 0:
    unusual_character_risk_level = "Low"
elif unusual_character_percentage < 10:
    unusual_character_risk_level = "Medium"
else:
    unusual_character_risk_level = "High"

unusual_character_risk_note = (
    f"{total_unusual_character_examples} prompts contain unusual characters "
    f"({round(unusual_character_percentage, 2)}% of the combined dataset). "
    "Most examples are INJECTION, but a small number of SAFE prompts also contain unusual or invisible characters."
)

data_quality_risk_rows = [
    row for row in data_quality_risk_rows
    if row["risk_area"] != "unusual_characters"
]

data_quality_risk_rows.append(
    {
        "risk_area": "unusual_characters",
        "risk_level": unusual_character_risk_level,
        "evidence": (
            f"Unusual-character prompts: {total_unusual_character_examples}/{total_dataset_rows} "
            f"({round(unusual_character_percentage, 2)}%)."
        ),
        "part_2_action": (
            "Keep unusual-character validation in data preparation. "
            "Do not automatically remove all unusual characters because some may represent adversarial prompt features. "
            "Review zero-width/invisible characters carefully before final preprocessing."
        ),
        "notes": unusual_character_risk_note
    }
)

print("Unusual-character risk summary by label:")
display(unusual_character_risk_summary_by_label_df)

print("\nUnusual-character risk summary by split:")
display(unusual_character_risk_summary_by_split_df)

print("\nUnusual-character type summary:")
display(unusual_character_type_summary_df)

print("\nUnusual-character final summary:")
display(pd.DataFrame([data_quality_risk_rows[-1]]))

In [ ]:
# 14-Save data quality risk summary

# Add possible label-noise risk summary before final saving
if "possible_label_noise_examples_df" in globals():
    possible_label_noise_candidate_count = len(possible_label_noise_examples_df)
else:
    possible_label_noise_candidate_count = 0

label_noise_risk_level = "Low" if possible_label_noise_candidate_count == 0 else "Medium"

data_quality_risk_rows = [
    row for row in data_quality_risk_rows
    if row["risk_area"] != "possible_label_noise"
]

data_quality_risk_rows.append(
    {
        "risk_area": "possible_label_noise",
        "risk_level": label_noise_risk_level,
        "evidence": f"Possible label-noise review candidates identified: {possible_label_noise_candidate_count}",
        "part_2_action": (
            "Manually review candidate examples during Part 2 data preparation. "
            "Do not automatically relabel or remove examples without justification."
        ),
        "notes": (
            "Possible label-noise candidates include very short INJECTION prompts, SAFE prompts with command/code-like signals, "
            "and SAFE prompts with unusual or invisible characters."
            if possible_label_noise_candidate_count > 0
            else "No possible label-noise candidates were identified."
        )
    }
)

risk_order = {
    "missing_values": 1,
    "empty_invalid_text": 2,
    "duplicates": 3,
    "train_test_leakage": 4,
    "class_imbalance": 5,
    "long_text_truncation": 6,
    "unusual_characters": 7,
    "possible_label_noise": 8
}

data_quality_risk_summary_df = pd.DataFrame(data_quality_risk_rows)

data_quality_risk_summary_df["risk_order"] = data_quality_risk_summary_df["risk_area"].map(risk_order)

data_quality_risk_summary_df = data_quality_risk_summary_df.sort_values(
    by="risk_order"
).drop(columns=["risk_order"]).reset_index(drop=True)

data_quality_risk_summary_path = TABLES_DIR / "data_quality_risk_summary.csv"

data_quality_risk_summary_df.to_csv(data_quality_risk_summary_path, index=False)

print("Data quality risk summary saved to:", data_quality_risk_summary_path)
print("File exists:", data_quality_risk_summary_path.exists())

print("\nFinal data quality risk summary:")
display(data_quality_risk_summary_df)

Essential Visualisation

#Part-12: Essential Visualisation

In [ ]:
# 1-Create class distribution by split figure

FIGURES_DIR = TABLES_DIR.parent / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

class_distribution_plot_df = (
    combined_eda_df
    .groupby(["split", "label"])
    .size()
    .reset_index(name="count")
)

class_distribution_plot_df["label_name"] = class_distribution_plot_df["label"].map(label_mapping)

class_distribution_pivot_df = class_distribution_plot_df.pivot(
    index="split",
    columns="label_name",
    values="count"
).fillna(0)

class_distribution_pivot_df = class_distribution_pivot_df[["SAFE", "INJECTION"]]

ax = class_distribution_pivot_df.plot(
    kind="bar",
    figsize=(8, 5),
    rot=0
)

ax.set_title("Class Distribution by Dataset Split")
ax.set_xlabel("Dataset split")
ax.set_ylabel("Number of prompts")
ax.legend(title="Class")

for container in ax.containers:
    ax.bar_label(container, label_type="edge", padding=3)

plt.tight_layout()

class_distribution_figure_path = FIGURES_DIR / "class_distribution_by_split.png"

plt.savefig(class_distribution_figure_path, dpi=300, bbox_inches="tight")
plt.show()

print("Class distribution figure saved to:", class_distribution_figure_path)
print("File exists:", class_distribution_figure_path.exists())

print("\nClass distribution table used for plotting:")
display(class_distribution_plot_df)

In [ ]:
# 2-Create character-length distribution figure

if "FIGURES_DIR" not in globals():
    FIGURES_DIR = TABLES_DIR.parent / "figures"
    FIGURES_DIR.mkdir(parents=True, exist_ok=True)

if "text_length_analysis_df" not in globals():
    text_length_analysis_df = pd.read_csv(
        TABLES_DIR / "text_length_analysis_with_token_lengths.csv"
    )

character_length_safe = text_length_analysis_df[
    text_length_analysis_df["label"] == 0
]["character_length"]

character_length_injection = text_length_analysis_df[
    text_length_analysis_df["label"] == 1
]["character_length"]

plt.figure(figsize=(9, 5))

plt.hist(
    [character_length_safe, character_length_injection],
    bins=30,
    label=["SAFE", "INJECTION"]
)

plt.title("Character-Length Distribution by Class")
plt.xlabel("Character length")
plt.ylabel("Number of prompts")
plt.legend(title="Class")

plt.tight_layout()

character_length_distribution_figure_path = FIGURES_DIR / "character_length_distribution_by_class.png"

plt.savefig(character_length_distribution_figure_path, dpi=300, bbox_inches="tight")
plt.show()

print("Character-length distribution figure saved to:", character_length_distribution_figure_path)
print("File exists:", character_length_distribution_figure_path.exists())

print("\nCharacter-length summary used for plotting:")
display(
    text_length_analysis_df
    .groupby("label")
    .agg(
        prompt_count=("text", "count"),
        min_character_length=("character_length", "min"),
        median_character_length=("character_length", "median"),
        mean_character_length=("character_length", "mean"),
        max_character_length=("character_length", "max")
    )
    .reset_index()
    .assign(label_name=lambda df: df["label"].map(label_mapping))
    [
        [
            "label",
            "label_name",
            "prompt_count",
            "min_character_length",
            "median_character_length",
            "mean_character_length",
            "max_character_length"
        ]
    ]
)

In [ ]:
# 3-Create word-count distribution figure

if "FIGURES_DIR" not in globals():
    FIGURES_DIR = TABLES_DIR.parent / "figures"
    FIGURES_DIR.mkdir(parents=True, exist_ok=True)

if "text_length_analysis_df" not in globals():
    text_length_analysis_df = pd.read_csv(
        TABLES_DIR / "text_length_analysis_with_token_lengths.csv"
    )

word_count_safe = text_length_analysis_df[
    text_length_analysis_df["label"] == 0
]["word_count"]

word_count_injection = text_length_analysis_df[
    text_length_analysis_df["label"] == 1
]["word_count"]

plt.figure(figsize=(9, 5))

plt.hist(
    [word_count_safe, word_count_injection],
    bins=30,
    label=["SAFE", "INJECTION"]
)

plt.title("Word-Count Distribution by Class")
plt.xlabel("Word count")
plt.ylabel("Number of prompts")
plt.legend(title="Class")

plt.tight_layout()

word_count_distribution_figure_path = FIGURES_DIR / "word_count_distribution_by_class.png"

plt.savefig(word_count_distribution_figure_path, dpi=300, bbox_inches="tight")
plt.show()

print("Word-count distribution figure saved to:", word_count_distribution_figure_path)
print("File exists:", word_count_distribution_figure_path.exists())

print("\nWord-count summary used for plotting:")
display(
    text_length_analysis_df
    .groupby("label")
    .agg(
        prompt_count=("text", "count"),
        min_word_count=("word_count", "min"),
        median_word_count=("word_count", "median"),
        mean_word_count=("word_count", "mean"),
        max_word_count=("word_count", "max")
    )
    .reset_index()
    .assign(label_name=lambda df: df["label"].map(label_mapping))
    [
        [
            "label",
            "label_name",
            "prompt_count",
            "min_word_count",
            "median_word_count",
            "mean_word_count",
            "max_word_count"
        ]
    ]
)

In [ ]:
# 4-Create token-length distribution figure

if "FIGURES_DIR" not in globals():
    FIGURES_DIR = TABLES_DIR.parent / "figures"
    FIGURES_DIR.mkdir(parents=True, exist_ok=True)

if "text_length_analysis_df" not in globals():
    text_length_analysis_df = pd.read_csv(
        TABLES_DIR / "text_length_analysis_with_token_lengths.csv"
    )

token_length_columns = {
    "DistilBERT": "distilbert_token_length",
    "RoBERTa": "roberta_token_length",
    "SecBERT": "secbert_token_length"
}

token_length_values = [
    text_length_analysis_df[column_name]
    for column_name in token_length_columns.values()
]

plt.figure(figsize=(9, 5))

plt.hist(
    token_length_values,
    bins=30,
    label=list(token_length_columns.keys())
)

plt.axvline(
    x=512,
    linestyle="--",
    linewidth=1.5,
    label="512-token limit"
)

plt.title("Token-Length Distribution by Transformer Tokenizer")
plt.xlabel("Token length")
plt.ylabel("Number of prompts")
plt.legend(title="Tokenizer")

plt.tight_layout()

token_length_distribution_figure_path = FIGURES_DIR / "token_length_distribution_by_tokenizer.png"

plt.savefig(token_length_distribution_figure_path, dpi=300, bbox_inches="tight")
plt.show()

print("Token-length distribution figure saved to:", token_length_distribution_figure_path)
print("File exists:", token_length_distribution_figure_path.exists())

print("\nToken-length summary used for plotting:")

token_length_summary_rows = []

for model_name, column_name in token_length_columns.items():
    token_length_summary_rows.append(
        {
            "model": model_name,
            "prompt_count": len(text_length_analysis_df),
            "min_token_length": text_length_analysis_df[column_name].min(),
            "median_token_length": text_length_analysis_df[column_name].median(),
            "mean_token_length": round(text_length_analysis_df[column_name].mean(), 2),
            "max_token_length": text_length_analysis_df[column_name].max(),
            "prompts_above_512": int((text_length_analysis_df[column_name] > 512).sum())
        }
    )

token_length_distribution_summary_df = pd.DataFrame(token_length_summary_rows)

display(token_length_distribution_summary_df)

In [ ]:
# 5-Create length-by-label comparison figure

if "FIGURES_DIR" not in globals():
    FIGURES_DIR = TABLES_DIR.parent / "figures"
    FIGURES_DIR.mkdir(parents=True, exist_ok=True)

if "text_length_analysis_df" not in globals():
    text_length_analysis_df = pd.read_csv(
        TABLES_DIR / "text_length_analysis_with_token_lengths.csv"
    )

length_by_label_df = text_length_analysis_df.copy()
length_by_label_df["label_name"] = length_by_label_df["label"].map(label_mapping)

length_by_label_summary_df = (
    length_by_label_df
    .groupby(["label", "label_name"])
    .agg(
        median_character_length=("character_length", "median"),
        median_word_count=("word_count", "median"),
        median_distilbert_tokens=("distilbert_token_length", "median"),
        median_roberta_tokens=("roberta_token_length", "median"),
        median_secbert_tokens=("secbert_token_length", "median")
    )
    .reset_index()
)

length_by_label_plot_df = length_by_label_summary_df.set_index("label_name")[
    [
        "median_character_length",
        "median_word_count",
        "median_distilbert_tokens",
        "median_roberta_tokens",
        "median_secbert_tokens"
    ]
]

ax = length_by_label_plot_df.plot(
    kind="bar",
    figsize=(10, 5),
    rot=0
)

ax.set_title("Median Length Comparison by Class")
ax.set_xlabel("Class")
ax.set_ylabel("Median length")
ax.legend(
    [
        "Characters",
        "Words",
        "DistilBERT tokens",
        "RoBERTa tokens",
        "SecBERT tokens"
    ],
    title="Length measure"
)

for container in ax.containers:
    ax.bar_label(container, label_type="edge", padding=3, fmt="%.0f")

plt.tight_layout()

length_by_label_comparison_figure_path = FIGURES_DIR / "median_length_comparison_by_class.png"

plt.savefig(length_by_label_comparison_figure_path, dpi=300, bbox_inches="tight")
plt.show()

print("Length-by-label comparison figure saved to:", length_by_label_comparison_figure_path)
print("File exists:", length_by_label_comparison_figure_path.exists())

print("\nLength-by-label summary used for plotting:")
display(length_by_label_summary_df)